# Global Automotive Investment Database — Block 4

## European filing architecture using direct API resource URLs

This version uses the `json_url`, `package_url`, and `viewer_url` fields already
returned by the filings.xbrl.org API.

The URLs are relative repository paths, so they are converted to absolute URLs
with `urljoin`. Retrieval proceeds in this order:

1. direct xBRL-JSON;
2. published filing package as fallback;
3. viewer URL retained for diagnostics only.

In [1]:
# 1. INSTALL / IMPORT DEPENDENCIES
# ------------------------------------------------

!pip -q install pandas numpy requests tqdm pyarrow

from __future__ import annotations

import hashlib
import json
import re
import time
import zipfile

from io import BytesIO

from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Optional
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests

from tqdm.auto import tqdm

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 260)

In [2]:
# 2. USER SETTINGS, DIRECTORIES AND UPSTREAM INPUTS
# ------------------------------------------------

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy"
    )
else:
    PROJECT_ROOT = Path("/content/global_automotive_investment_database")

DATA_ROOT = PROJECT_ROOT / "data"

BLOCK_2_OUTPUT_DIR = DATA_ROOT / "interim" / "block_2"
BLOCK_2_MANIFEST_PATH = BLOCK_2_OUTPUT_DIR / "block_2_manifest.json"

BLOCK_3_OUTPUT_DIR = DATA_ROOT / "interim" / "block_3"
BLOCK_3_MANIFEST_PATH = BLOCK_3_OUTPUT_DIR / "block_3_manifest.json"

BLOCK_4_OUTPUT_DIR = DATA_ROOT / "interim" / "block_4"
BLOCK_4_MANIFEST_PATH = BLOCK_4_OUTPUT_DIR / "block_4_manifest.json"

EUROPE_RAW_DIR = DATA_ROOT / "raw" / "europe_filings"
EUROPE_API_CACHE_DIR = EUROPE_RAW_DIR / "filings_xbrl_api"
EUROPE_FACT_CACHE_DIR = EUROPE_RAW_DIR / "xbrl_json"

for directory in [
    BLOCK_4_OUTPUT_DIR,
    EUROPE_RAW_DIR,
    EUROPE_API_CACHE_DIR,
    EUROPE_FACT_CACHE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Public discovery layer.
FILINGS_XBRL_API_BASE = "https://filings.xbrl.org/api"
FILINGS_XBRL_BASE = "https://filings.xbrl.org"

REQUEST_INTERVAL_SECONDS = 0.20
REQUEST_TIMEOUT_SECONDS = 90
MAX_RETRIES = 5
PAGE_SIZE = 200

# None means no artificial limit.
MAX_API_PAGES_PER_COUNTRY = None
MAX_FILINGS_TO_DOWNLOAD = None

# Point-in-time cutoff. None means all currently discoverable filings.
AS_OF_DATE = None  # example: "2025-12-31"

DOWNLOAD_XBRL_JSON = True
PERSIST_BLOCK_4_OUTPUTS = True
OVERWRITE_PERSISTED_OUTPUTS = True

# Countries treated as European for this module.
EUROPE_COUNTRY_CODES = {
    "AT", "BE", "BG", "CH", "CY", "CZ", "DE", "DK", "EE", "ES", "FI",
    "FR", "GB", "GR", "HR", "HU", "IE", "IS", "IT", "LI", "LT", "LU",
    "LV", "MT", "NL", "NO", "PL", "PT", "RO", "SE", "SI", "SK",
}


def load_manifest_tables(
    manifest_path: Path,
    required_table_names: set[str],
) -> tuple[dict[str, pd.DataFrame], dict]:

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Required upstream manifest not found: {manifest_path}"
        )

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    records = {
        item["table_name"]: item
        for item in manifest.get("tables", [])
    }

    missing = required_table_names.difference(records)

    if missing:
        raise RuntimeError(
            f"Manifest {manifest_path.name} is missing tables: {sorted(missing)}"
        )

    loaded = {}

    for table_name in sorted(required_table_names):
        table_path = Path(records[table_name]["path"])

        if not table_path.exists():
            raise FileNotFoundError(
                f"Manifest entry exists but file is missing: {table_path}"
            )

        loaded[table_name] = pd.read_parquet(table_path)

    return loaded, manifest


block_2_inputs, block_2_manifest = load_manifest_tables(
    BLOCK_2_MANIFEST_PATH,
    {
        "security_master_df",
        "issuer_master_df",
        "security_identifier_history_df",
        "source_security_bridge_df",
    },
)

security_master_df = block_2_inputs["security_master_df"]
issuer_master_df = block_2_inputs["issuer_master_df"]
security_identifier_history_df = block_2_inputs[
    "security_identifier_history_df"
]
source_security_bridge_df = block_2_inputs["source_security_bridge_df"]

# Block 3's standard concept dictionary is reused where available.
block_3_inputs, block_3_manifest = load_manifest_tables(
    BLOCK_3_MANIFEST_PATH,
    {"sec_standard_concept_dictionary_df"},
)

sec_standard_concept_dictionary_df = block_3_inputs[
    "sec_standard_concept_dictionary_df"
]

print("Loaded upstream data without executing earlier notebooks.")
for name, dataframe in {**block_2_inputs, **block_3_inputs}.items():
    print(f"  {name}: {len(dataframe):,} rows × {len(dataframe.columns):,} columns")

print("\nBlock 4 output directory:", BLOCK_4_OUTPUT_DIR)

Mounted at /content/drive
Loaded upstream data without executing earlier notebooks.
  issuer_master_df: 566 rows × 16 columns
  security_identifier_history_df: 22,368 rows × 11 columns
  security_master_df: 512 rows × 34 columns
  source_security_bridge_df: 7,968 rows × 36 columns
  sec_standard_concept_dictionary_df: 147 rows × 10 columns

Block 4 output directory: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_4


In [3]:
# 3. EUROPEAN NATIONAL-SYSTEM REGISTRY
# ------------------------------------------------

# This is an architectural control table, not a claim that every system exposes
# a stable public API. Connector status remains explicit and auditable.

EUROPE_FILING_SYSTEMS = [
    ("AT", "Austria", "OeKB Issuer Information", "OAM", "FEDERATED"),
    ("BE", "Belgium", "FSMA / STORI", "OAM", "FEDERATED"),
    ("BG", "Bulgaria", "Financial Supervision Commission", "OAM", "FEDERATED"),
    ("CH", "Switzerland", "SIX Exchange Regulation / issuer sources", "NATIONAL", "FEDERATED"),
    ("CY", "Cyprus", "CySEC", "OAM", "FEDERATED"),
    ("CZ", "Czech Republic", "Czech National Bank", "OAM", "FEDERATED"),
    ("DE", "Germany", "Unternehmensregister", "OAM", "NATIONAL_CONNECTOR_REQUIRED"),
    ("DK", "Denmark", "Danish FSA / OAM", "OAM", "FEDERATED"),
    ("EE", "Estonia", "Nasdaq Baltic / national register", "NATIONAL", "FEDERATED"),
    ("ES", "Spain", "CNMV", "OAM", "FEDERATED"),
    ("FI", "Finland", "Finanssivalvonta / Nasdaq", "OAM", "FEDERATED"),
    ("FR", "France", "AMF / INFO-FINANCIERE", "OAM", "FEDERATED"),
    ("GB", "United Kingdom", "FCA National Storage Mechanism", "NSM", "FEDERATED"),
    ("GR", "Greece", "Hellenic Capital Market Commission", "OAM", "FEDERATED"),
    ("HR", "Croatia", "HANFA / SRPI", "OAM", "FEDERATED"),
    ("HU", "Hungary", "Central Bank of Hungary", "OAM", "FEDERATED"),
    ("IE", "Ireland", "Central Bank of Ireland", "OAM", "NATIONAL_CONNECTOR_REQUIRED"),
    ("IS", "Iceland", "Central Bank of Iceland / OAM", "OAM", "FEDERATED"),
    ("IT", "Italy", "CONSOB / authorised storage systems", "OAM", "FEDERATED"),
    ("LI", "Liechtenstein", "FMA Liechtenstein", "OAM", "FEDERATED"),
    ("LT", "Lithuania", "Bank of Lithuania / Nasdaq Baltic", "OAM", "FEDERATED"),
    ("LU", "Luxembourg", "Luxembourg Stock Exchange OAM", "OAM", "FEDERATED"),
    ("LV", "Latvia", "Bank of Latvia / Nasdaq Baltic", "OAM", "FEDERATED"),
    ("MT", "Malta", "Malta Financial Services Authority", "OAM", "FEDERATED"),
    ("NL", "Netherlands", "AFM", "OAM", "FEDERATED"),
    ("NO", "Norway", "Finanstilsynet / Oslo Børs", "NATIONAL", "FEDERATED"),
    ("PL", "Poland", "KNF", "OAM", "FEDERATED"),
    ("PT", "Portugal", "CMVM", "OAM", "FEDERATED"),
    ("RO", "Romania", "Financial Supervisory Authority", "OAM", "FEDERATED"),
    ("SE", "Sweden", "Finansinspektionen", "OAM", "FEDERATED"),
    ("SI", "Slovenia", "Securities Market Agency", "OAM", "FEDERATED"),
    ("SK", "Slovakia", "National Bank of Slovakia", "OAM", "FEDERATED"),
]

europe_filing_system_registry_df = pd.DataFrame(
    EUROPE_FILING_SYSTEMS,
    columns=[
        "country_code",
        "country_name",
        "system_name",
        "system_type",
        "connector_status",
    ],
)

europe_filing_system_registry_df["discovery_layer"] = "filings.xbrl.org"
europe_filing_system_registry_df["official_source_priority"] = True
europe_filing_system_registry_df["notes"] = pd.NA

display(europe_filing_system_registry_df)

,country_code,country_name,system_name,system_type,connector_status,discovery_layer,official_source_priority,notes
0,AT,Austria,OeKB Issuer Information,OAM,FEDERATED,filings.xbrl.org,True,<NA>
1,BE,Belgium,FSMA / STORI,OAM,FEDERATED,filings.xbrl.org,True,<NA>
2,BG,Bulgaria,Financial Supervision Commission,OAM,FEDERATED,filings.xbrl.org,True,<NA>
3,CH,Switzerland,SIX Exchange Regulation / issuer sources,NATIONAL,FEDERATED,filings.xbrl.org,True,<NA>
4,CY,Cyprus,CySEC,OAM,FEDERATED,filings.xbrl.org,True,<NA>
5,CZ,Czech Republic,Czech National Bank,OAM,FEDERATED,filings.xbrl.org,True,<NA>
6,DE,Germany,Unternehmensregister,OAM,NATIONAL_CONNECTOR_REQUIRED,filings.xbrl.org,True,<NA>
7,DK,Denmark,Danish FSA / OAM,OAM,FEDERATED,filings.xbrl.org,True,<NA>
8,EE,Estonia,Nasdaq Baltic / national register,NATIONAL,FEDERATED,filings.xbrl.org,True,<NA>
9,ES,Spain,CNMV,OAM,FEDERATED,filings.xbrl.org,True,<NA>


In [4]:
# 4. BUILD THE EUROPEAN SECURITY AND ISSUER UNIVERSE
# ------------------------------------------------

def first_existing_column(
    dataframe: pd.DataFrame,
    candidates: Iterable[str],
) -> Optional[str]:
    return next(
        (column for column in candidates if column in dataframe.columns),
        None,
    )


def clean_text(value):
    if pd.isna(value):
        return pd.NA
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value if value else pd.NA


def clean_country(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().upper()
    return value if len(value) == 2 else pd.NA


def clean_lei(value):
    if pd.isna(value):
        return pd.NA
    value = re.sub(r"[^A-Z0-9]", "", str(value).upper())
    return value if len(value) == 20 else pd.NA


def extract_identifier_history(
    history: pd.DataFrame,
    identifier_types: set[str],
) -> pd.DataFrame:

    if history.empty:
        return pd.DataFrame()

    type_col = first_existing_column(
        history,
        ["identifier_type", "id_type", "type"],
    )
    value_col = first_existing_column(
        history,
        ["identifier_value", "id_value", "value"],
    )

    if type_col is None or value_col is None:
        return pd.DataFrame()

    subset = history[
        history[type_col].astype("string").str.upper().isin(identifier_types)
    ].copy()

    return subset


country_col = first_existing_column(
    security_master_df,
    ["country", "issuer_country", "domicile_country"],
)
lei_col = first_existing_column(
    security_master_df,
    ["lei", "issuer_lei"],
)
ticker_col = first_existing_column(
    security_master_df,
    ["ticker", "primary_ticker", "source_ticker"],
)
issuer_name_col = first_existing_column(
    security_master_df,
    ["issuer_name", "security_name", "name"],
)

europe_security_universe_df = pd.DataFrame(index=security_master_df.index)

for target, source in {
    "security_id": first_existing_column(security_master_df, ["security_id"]),
    "issuer_id": first_existing_column(security_master_df, ["issuer_id"]),
}.items():
    europe_security_universe_df[target] = (
        security_master_df[source] if source else pd.NA
    )

europe_security_universe_df["issuer_name"] = (
    security_master_df[issuer_name_col].map(clean_text)
    if issuer_name_col else pd.NA
)
europe_security_universe_df["ticker"] = (
    security_master_df[ticker_col].map(clean_text)
    if ticker_col else pd.NA
)
europe_security_universe_df["country"] = (
    security_master_df[country_col].map(clean_country)
    if country_col else pd.NA
)
europe_security_universe_df["lei"] = (
    security_master_df[lei_col].map(clean_lei)
    if lei_col else pd.NA
)

# Add LEIs from long-form identifier history where the master lacks them.
lei_history = extract_identifier_history(
    security_identifier_history_df,
    {"LEI", "ISSUER_LEI"},
)

if not lei_history.empty:
    history_security_col = first_existing_column(
        lei_history,
        ["security_id"],
    )
    history_value_col = first_existing_column(
        lei_history,
        ["identifier_value", "id_value", "value"],
    )

    if history_security_col and history_value_col:
        history_lei_map = (
            lei_history[[history_security_col, history_value_col]]
            .assign(
                history_lei=lambda frame: frame[history_value_col].map(clean_lei)
            )
            .dropna(subset=["history_lei"])
            .drop_duplicates(history_security_col)
            .rename(columns={history_security_col: "security_id"})
            [["security_id", "history_lei"]]
        )

        europe_security_universe_df = europe_security_universe_df.merge(
            history_lei_map,
            on="security_id",
            how="left",
        )

        europe_security_universe_df["lei"] = (
            europe_security_universe_df["lei"]
            .fillna(europe_security_universe_df["history_lei"])
        )

        europe_security_universe_df = europe_security_universe_df.drop(
            columns=["history_lei"]
        )

europe_security_universe_df = (
    europe_security_universe_df[
        europe_security_universe_df["country"].isin(EUROPE_COUNTRY_CODES)
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

europe_issuer_universe_df = (
    europe_security_universe_df[
        ["issuer_id", "issuer_name", "country", "lei"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"European securities: {len(europe_security_universe_df):,}\n"
    f"European issuers: {len(europe_issuer_universe_df):,}\n"
    f"Issuers with valid LEI: {europe_issuer_universe_df['lei'].notna().mean():.2%}"
)

display(europe_issuer_universe_df.head(30))

# ------------------------------------------------------------
# ISSUER-CENTRIC EUROPEAN CONTRACTS
# ------------------------------------------------------------

def clean_lei(value):
    if pd.isna(value):
        return pd.NA

    text = re.sub(
        r"\s+",
        "",
        str(value),
    ).upper()

    return (
        text
        if re.fullmatch(
            r"[A-Z0-9]{20}",
            text,
        )
        else pd.NA
    )


europe_security_universe_df[
    "lei"
] = europe_security_universe_df[
    "lei"
].map(clean_lei)

europe_issuer_universe_df[
    "lei"
] = europe_issuer_universe_df[
    "lei"
].map(clean_lei)

europe_economic_issuer_universe_df = (
    europe_issuer_universe_df.copy()
)

europe_issuer_security_universe_df = (
    europe_security_universe_df.copy()
)


europe_lei_issuer_bridge_candidates_df = (
    europe_economic_issuer_universe_df[
        [
            column
            for column in [
                "lei",
                "issuer_id",
                "issuer_name",
                "country",
            ]
            if column
            in europe_economic_issuer_universe_df.columns
        ]
    ]
    .dropna(
        subset=[
            "lei",
            "issuer_id",
        ]
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

europe_lei_bridge_quality_df = (
    europe_lei_issuer_bridge_candidates_df
    .groupby(
        "lei",
        dropna=False,
    )
    .agg(
        issuer_id_count=(
            "issuer_id",
            "nunique",
        ),
        issuer_name_count=(
            "issuer_name",
            "nunique",
        ),
        country_count=(
            "country",
            "nunique",
        ),
    )
    .reset_index()
)

europe_lei_issuer_bridge_candidates_df = (
    europe_lei_issuer_bridge_candidates_df
    .merge(
        europe_lei_bridge_quality_df,
        on="lei",
        how="left",
        validate="m:1",
    )
)

europe_lei_issuer_bridge_df = (
    europe_lei_issuer_bridge_candidates_df[
        europe_lei_issuer_bridge_candidates_df[
            "issuer_id_count"
        ].eq(1)
    ]
    .sort_values(
        [
            "lei",
            "issuer_id",
        ],
        na_position="last",
    )
    .drop_duplicates(
        "lei",
        keep="first",
    )
    .reset_index(drop=True)
)

europe_lei_issuer_conflicts_df = (
    europe_lei_issuer_bridge_candidates_df[
        ~europe_lei_issuer_bridge_candidates_df[
            "issuer_id_count"
        ].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)

europe_preferred_accounting_source_df = (
    europe_lei_issuer_bridge_df[
        [
            column
            for column in [
                "issuer_id",
                "issuer_name",
                "lei",
                "country",
            ]
            if column
            in europe_lei_issuer_bridge_df.columns
        ]
    ]
    .rename(
        columns={
            "lei": "preferred_source_entity_id",
        }
    )
    .assign(
        preferred_source_system=(
            "ESEF_UKSEF_OR_NATIONAL_OAM"
        ),
        preferred_source_region="EUROPE",
        source_confidence=1.0,
        selection_basis=(
            "AUTHORITATIVE_LEI_TO_ISSUER_BRIDGE"
        ),
    )
    .reset_index(drop=True)
)

europe_entity_relationship_graph_df = pd.DataFrame(
    columns=[
        "from_entity_id",
        "to_entity_id",
        "relationship_type",
        "effective_start",
        "effective_end",
        "confidence",
        "source_system",
        "source_region",
    ]
)

print(
    "Confirmed LEI-to-issuer mappings:",
    len(
        europe_lei_issuer_bridge_df
    ),
)
print(
    "Conflicted LEIs:",
    len(
        europe_lei_issuer_conflicts_df
    ),
)


European securities: 81
European issuers: 77
Issuers with valid LEI: 87.01%


,issuer_id,issuer_name,country,lei
0,GAI_B4458F7BDC67FFDB71E4,TI Fluid Systems PLC,GB,5493001T9RXVD6OAWY46
1,GAI_5CD5347816CF9C46E768,Aston Martin Lagonda Global Ho,GB,213800167WOVOK5ZC776
2,GAI_852D4621135BE55DE521,Piaggio & C SpA,IT,8156000256C2431C2E92
3,GAI_B0B2DA0C6FF143ACE220,Continental AG,DE,529900A7YD9C0LLXM621
4,GAI_A5146E658A5377F41954,Stellantis N.V.,IT,549300LKT9PW7ZIBDF31
5,GAI_C411BA8AE70FBDAD20FC,Volvo AB,SE,549300HGV012CNC8JD22
6,GAI_43EF3B109F226A6EF7BD,Volkswagen AG,DE,529900NNUPAGGOMPXZ31
7,GAI_CC2F304AD8F787FB579B,Melexis NV,BE,549300QRPSGOJRPUFO80
8,GAI_B815F09E4557E52DE14C,Wacker Chemie AG,DE,0NURKC5Q3CJYZPPK5046
9,GAI_667F4A3CB2EE2BFBFA37,Spectris PLC,GB,213800Z4CO2CZO3M3T10


Confirmed LEI-to-issuer mappings: 64
Conflicted LEIs: 0


In [5]:
# 5. PERSISTENT HTTP SESSION AND CACHE HELPERS
# ------------------------------------------------

session = requests.Session()
session.headers.update({
    "User-Agent": "Global Automotive Investment Database research client",
    "Accept": "application/vnd.api+json, application/json",
})


def cache_key(url: str, params: Optional[dict] = None) -> str:
    payload = json.dumps(
        {"url": url, "params": params or {}},
        sort_keys=True,
        default=str,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def request_json_cached(
    url: str,
    params: Optional[dict] = None,
    cache_dir: Path = EUROPE_API_CACHE_DIR,
    force_refresh: bool = False,
) -> tuple[dict, dict]:

    cache_path = cache_dir / f"{cache_key(url, params)}.json"

    if cache_path.exists() and not force_refresh:
        with cache_path.open("r", encoding="utf-8") as file:
            return json.load(file), {
                "url": url,
                "status": "CACHE_HIT",
                "cache_path": str(cache_path),
            }

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )

            response.raise_for_status()
            payload = response.json()

            with cache_path.open("w", encoding="utf-8") as file:
                json.dump(payload, file)

            time.sleep(REQUEST_INTERVAL_SECONDS)

            return payload, {
                "url": response.url,
                "status": "DOWNLOADED",
                "http_status": response.status_code,
                "cache_path": str(cache_path),
                "attempt": attempt,
            }

        except Exception as exc:
            last_error = repr(exc)
            time.sleep(min(2 ** attempt, 20))

    raise RuntimeError(
        f"Failed after {MAX_RETRIES} attempts: {url}; {last_error}"
    )

In [6]:
# 6. DISCOVER ESEF / UKSEF FILINGS BY COUNTRY
# ------------------------------------------------

def flatten_jsonapi_resource(
    resource: dict,
    included_entities: dict[str, dict],
) -> dict:

    attributes = resource.get("attributes", {}) or {}
    relationships = resource.get("relationships", {}) or {}
    links = resource.get("links", {}) or {}

    row = {
        "filing_resource_id": resource.get("id"),
        "resource_type": resource.get("type"),
        **attributes,
    }

    row["resource_self_url"] = (
        links.get("self")
        if isinstance(links, dict)
        else pd.NA
    )

    entity_rel = relationships.get("entity", {})
    entity_data = entity_rel.get("data") if isinstance(entity_rel, dict) else None

    if isinstance(entity_data, dict):
        entity_id = entity_data.get("id")
        row["entity_resource_id"] = entity_id

        entity = included_entities.get(entity_id, {})
        entity_attributes = entity.get("attributes", {}) or {}

        for key, value in entity_attributes.items():
            row[f"entity_{key}"] = value

    return row


def fetch_country_filings(country_code: str) -> tuple[pd.DataFrame, pd.DataFrame]:

    rows = []
    log_rows = []
    page_number = 1

    while True:
        params = {
            "filter[country]": country_code,
            "include": "entity",
            "sort": "-processed",
            "page[size]": PAGE_SIZE,
            "page[number]": page_number,
        }

        payload, log = request_json_cached(
            f"{FILINGS_XBRL_API_BASE}/filings",
            params=params,
        )

        log.update({
            "country": country_code,
            "page_number": page_number,
        })
        log_rows.append(log)

        included_entities = {
            item.get("id"): item
            for item in payload.get("included", [])
            if item.get("type") == "entity"
        }

        page_data = payload.get("data", [])

        rows.extend(
            flatten_jsonapi_resource(item, included_entities)
            for item in page_data
        )

        links = payload.get("links", {}) or {}
        next_link = links.get("next")

        if not next_link or not page_data:
            break

        page_number += 1

        if (
            MAX_API_PAGES_PER_COUNTRY is not None
            and page_number > int(MAX_API_PAGES_PER_COUNTRY)
        ):
            break

    return pd.DataFrame(rows), pd.DataFrame(log_rows)


countries_to_query = sorted(
    set(europe_issuer_universe_df["country"].dropna())
)

filing_frames = []
discovery_log_frames = []

for country_code in tqdm(
    countries_to_query,
    desc="Discovering European filings",
):

    country_filings, country_log = fetch_country_filings(country_code)

    if not country_filings.empty:
        filing_frames.append(country_filings)

    if not country_log.empty:
        discovery_log_frames.append(country_log)

europe_filings_discovered_df = (
    pd.concat(filing_frames, ignore_index=True)
    if filing_frames
    else pd.DataFrame()
)

europe_filing_discovery_log_df = (
    pd.concat(discovery_log_frames, ignore_index=True)
    if discovery_log_frames
    else pd.DataFrame()
)

print(f"Discovered filing records: {len(europe_filings_discovered_df):,}")
display(europe_filings_discovered_df.head())

Discovering European filings:   0%|          | 0/13 [00:00<?, ?it/s]

Discovered filing records: 10,281


,filing_resource_id,resource_type,viewer_url,inconsistency_count,package_url,warning_count,period_end,report_url,processed,fxo_id,country,date_added,sha256,json_url,error_count,resource_self_url,entity_resource_id,entity_name,entity_identifier
0,25155,filing,/529900EVOKN4LCCD9321/2026-03-31/ESEF/AT/1/ats...,0,/529900EVOKN4LCCD9321/2026-03-31/ESEF/AT/1/529...,4,2026-03-31,/529900EVOKN4LCCD9321/2026-03-31/ESEF/AT/1/ats...,2026-07-08 11:33:43.230008,529900EVOKN4LCCD9321-2026-03-31-ESEF-AT-1,AT,2026-06-25 18:39:52.629402,c46ee884b0c17e8c4f27c3d8b61e62a7ef92af41e77e47...,/529900EVOKN4LCCD9321/2026-03-31/ESEF/AT/1/ats...,0,/api/filings/25155,2450,AT & S Austria Technologie & Systemtechnik Akt...,529900EVOKN4LCCD9321
1,25154,filing,/529900EVOKN4LCCD9321/2026-03-31/ESEF/AT/0/ats...,0,/529900EVOKN4LCCD9321/2026-03-31/ESEF/AT/0/529...,4,2026-03-31,/529900EVOKN4LCCD9321/2026-03-31/ESEF/AT/0/ats...,2026-07-08 11:33:25.634714,529900EVOKN4LCCD9321-2026-03-31-ESEF-AT-0,AT,2026-06-25 18:39:51.269299,269c2145815f58da3ce88d6605dfa3df4677d15ce4b638...,/529900EVOKN4LCCD9321/2026-03-31/ESEF/AT/0/ats...,0,/api/filings/25154,2450,AT & S Austria Technologie & Systemtechnik Akt...,529900EVOKN4LCCD9321
2,25134,filing,/529900ZAXBMQDIWPNB72/2026-03-31/ESEF/AT/0/voe...,0,/529900ZAXBMQDIWPNB72/2026-03-31/ESEF/AT/0/voe...,4,2026-03-31,/529900ZAXBMQDIWPNB72/2026-03-31/ESEF/AT/0/voe...,2026-06-11 17:27:19.773295,529900ZAXBMQDIWPNB72-2026-03-31-ESEF-AT-0,AT,2026-06-11 16:32:56.783848,54ffa3807629cf22d4bde4881e21a46de6dab8117ee1ec...,/529900ZAXBMQDIWPNB72/2026-03-31/ESEF/AT/0/voe...,0,/api/filings/25134,1968,voestalpine AG,529900ZAXBMQDIWPNB72
3,25133,filing,/391200WHND7OZEFNNL77/2026-03-31/ESEF/AT/0/fab...,0,/391200WHND7OZEFNNL77/2026-03-31/ESEF/AT/0/391...,0,2026-03-31,/391200WHND7OZEFNNL77/2026-03-31/ESEF/AT/0/fab...,2026-06-11 17:27:02.491999,391200WHND7OZEFNNL77-2026-03-31-ESEF-AT-0,AT,2026-06-11 16:32:56.179734,24a9a5c97bffe8ad373c990ec0f78c8bea865705ef3a89...,/391200WHND7OZEFNNL77/2026-03-31/ESEF/AT/0/391...,0,/api/filings/25133,610,Fabasoft AG,391200WHND7OZEFNNL77
4,25132,filing,/5299002NFQKOBT1E8569/2026-03-31/ESEF/AT/0/doc...,0,/5299002NFQKOBT1E8569/2026-03-31/ESEF/AT/0/529...,1,2026-03-31,/5299002NFQKOBT1E8569/2026-03-31/ESEF/AT/0/doc...,2026-06-11 17:26:37.113761,5299002NFQKOBT1E8569-2026-03-31-ESEF-AT-0,AT,2026-06-11 16:32:54.037181,4d7dc5e7fdd5e11641e9dfd99f36543f29aee70393f280...,/5299002NFQKOBT1E8569/2026-03-31/ESEF/AT/0/doc...,0,/api/filings/25132,2815,DO & CO Aktiengesellschaft,5299002NFQKOBT1E8569


In [7]:
# 7. NORMALISE FILING METADATA AND LINK FILINGS TO THE EUROPEAN UNIVERSE
# ------------------------------------------------

def coalesce_columns(
    dataframe: pd.DataFrame,
    candidates: list[str],
):
    available = [
        column
        for column in candidates
        if column in dataframe.columns
    ]

    if not available:
        return pd.Series(
            pd.NA,
            index=dataframe.index,
            dtype="object",
        )

    result = dataframe[
        available[0]
    ].copy()

    for column in available[1:]:
        result = result.fillna(
            dataframe[column]
        )

    return result


def normalise_date_series(series):
    return pd.to_datetime(
        series,
        errors="coerce",
        utc=True,
    )


if europe_filings_discovered_df.empty:
    europe_filing_metadata_df = pd.DataFrame()

else:
    source = (
        europe_filings_discovered_df.copy()
    )

    metadata = pd.DataFrame(
        index=source.index
    )

    metadata["filing_id"] = coalesce_columns(
        source,
        [
            "filing_resource_id",
            "id",
        ],
    )

    metadata["lei"] = coalesce_columns(
        source,
        [
            "entity_identifier",
            "entity_lei",
            "lei",
        ],
    ).map(clean_lei)

    metadata["entity_name"] = coalesce_columns(
        source,
        [
            "entity_name",
            "entity_legal_name",
            "name",
        ],
    ).map(clean_text)

    metadata["country"] = coalesce_columns(
        source,
        [
            "country",
            "entity_country",
        ],
    ).map(clean_country)

    metadata["filing_system"] = coalesce_columns(
        source,
        [
            "filing_system",
            "system",
        ],
    )

    metadata["reporting_date"] = (
        normalise_date_series(
            coalesce_columns(
                source,
                [
                    "reporting_date",
                    "period_end",
                    "report_date",
                ],
            )
        )
    )

    metadata["processed_datetime"] = (
        normalise_date_series(
            coalesce_columns(
                source,
                [
                    "processed",
                    "processed_at",
                    "date_added",
                ],
            )
        )
    )

    metadata["filing_date"] = (
        normalise_date_series(
            coalesce_columns(
                source,
                [
                    "filing_date",
                    "published",
                    "date",
                ],
            )
        )
    )

    metadata["available_datetime"] = (
        metadata["filing_date"]
        .fillna(
            metadata[
                "processed_datetime"
            ]
        )
    )

    metadata["availability_basis"] = np.select(
        [
            metadata[
                "filing_date"
            ].notna(),
            metadata[
                "processed_datetime"
            ].notna(),
        ],
        [
            "FILING_DATE",
            "AGGREGATOR_PROCESSED_TIMESTAMP",
        ],
        default="UNKNOWN",
    )

    metadata["language"] = coalesce_columns(
        source,
        [
            "language",
            "languages",
        ],
    )

    metadata["package_url"] = (
        coalesce_columns(
            source,
            [
                "package_url",
                "report_package",
                "download_url",
            ],
        )
    )

    metadata["json_url"] = coalesce_columns(
        source,
        [
            "json_url",
            "xbrl_json",
            "facts_url",
        ],
    )

    metadata["viewer_url"] = coalesce_columns(
        source,
        [
            "viewer_url",
            "viewer",
        ],
    )

    metadata["source_url"] = coalesce_columns(
        source,
        [
            "source_url",
            "resource_self_url",
        ],
    )

    metadata["raw_attributes_json"] = (
        source.apply(
            lambda row: json.dumps(
                row.dropna().to_dict(),
                default=str,
                sort_keys=True,
            ),
            axis=1,
        )
    )

    issuer_bridge = (
        europe_lei_issuer_bridge_df[
            [
                column
                for column in [
                    "issuer_id",
                    "issuer_name",
                    "country",
                    "lei",
                ]
                if column
                in europe_lei_issuer_bridge_df.columns
            ]
        ]
        .drop_duplicates(
            "lei"
        )
    )

    metadata = metadata.merge(
        issuer_bridge,
        on="lei",
        how="left",
        suffixes=(
            "",
            "_master",
        ),
        validate="m:1",
    )

    metadata["issuer_name"] = (
        metadata["issuer_name"]
        .fillna(
            metadata["entity_name"]
        )
    )

    metadata[
        "issuer_link_status"
    ] = np.where(
        metadata["issuer_id"].notna(),
        "LINKED",
        "UNRESOLVED_LEI_TO_ISSUER",
    )

    if AS_OF_DATE is not None:
        cutoff = pd.Timestamp(
            AS_OF_DATE,
            tz="UTC",
        )

        metadata = metadata[
            metadata[
                "available_datetime"
            ].isna()
            | (
                metadata[
                    "available_datetime"
                ]
                <= cutoff
            )
        ].copy()

    europe_filing_metadata_df = (
        metadata
        .drop_duplicates(
            subset=[
                "filing_id",
                "lei",
                "reporting_date",
                "filing_system",
            ]
        )
        .reset_index(drop=True)
    )


if europe_filing_metadata_df.empty:
    europe_matched_filings_df = (
        pd.DataFrame()
    )
    europe_unmatched_filings_df = (
        pd.DataFrame()
    )

else:
    europe_matched_filings_df = (
        europe_filing_metadata_df[
            europe_filing_metadata_df[
                "issuer_id"
            ].notna()
        ]
        .copy()
    )

    europe_unmatched_filings_df = (
        europe_filing_metadata_df[
            europe_filing_metadata_df[
                "issuer_id"
            ].isna()
        ]
        .copy()
    )


print(
    f"Normalised filings: "
    f"{len(europe_filing_metadata_df):,}\n"
    f"Matched to global issuer IDs: "
    f"{len(europe_matched_filings_df):,}"
)

display(
    europe_matched_filings_df.head(20)
)

Normalised filings: 10,281
Matched to global issuer IDs: 144


,filing_id,lei,entity_name,country,filing_system,reporting_date,processed_datetime,filing_date,available_datetime,availability_basis,language,package_url,json_url,viewer_url,source_url,raw_attributes_json,issuer_id,issuer_name,country_master,issuer_link_status
718,17780,529900F3AIQECS8ZSV61,UMICORE,BE,<NA>,2024-12-31 00:00:00+00:00,2025-04-15 16:22:12.606162+00:00,NaT,2025-04-15 16:22:12.606162+00:00,AGGREGATOR_PROCESSED_TIMESTAMP,<NA>,/529900F3AIQECS8ZSV61/2024-12-31/ESEF/BE/1/UMI...,/529900F3AIQECS8ZSV61/2024-12-31/ESEF/BE/1/UMI...,/529900F3AIQECS8ZSV61/2024-12-31/ESEF/BE/1/UMI...,/api/filings/17780,"{""country"": ""BE"", ""date_added"": ""2025-04-15 12...",GAI_93339BCC861531442C34,UMICORE SA,BE,LINKED
719,17779,529900F3AIQECS8ZSV61,UMICORE,BE,<NA>,2024-12-31 00:00:00+00:00,2025-04-15 16:21:53.313460+00:00,NaT,2025-04-15 16:21:53.313460+00:00,AGGREGATOR_PROCESSED_TIMESTAMP,<NA>,/529900F3AIQECS8ZSV61/2024-12-31/ESEF/BE/0/UMI...,/529900F3AIQECS8ZSV61/2024-12-31/ESEF/BE/0/UMI...,/529900F3AIQECS8ZSV61/2024-12-31/ESEF/BE/0/UMI...,/api/filings/17779,"{""country"": ""BE"", ""date_added"": ""2025-04-15 12...",GAI_93339BCC861531442C34,UMICORE SA,BE,LINKED
726,17772,549300MMVL80RTBP3O28,SOLVAY,BE,<NA>,2024-12-31 00:00:00+00:00,2025-04-15 16:16:43.416391+00:00,NaT,2025-04-15 16:16:43.416391+00:00,AGGREGATOR_PROCESSED_TIMESTAMP,<NA>,/549300MMVL80RTBP3O28/2024-12-31/ESEF/BE/2/549...,/549300MMVL80RTBP3O28/2024-12-31/ESEF/BE/2/549...,/549300MMVL80RTBP3O28/2024-12-31/ESEF/BE/2/549...,/api/filings/17772,"{""country"": ""BE"", ""date_added"": ""2025-04-15 12...",GAI_1D54C6F46FE0A91920AB,Solvay SA,BE,LINKED
727,17771,549300MMVL80RTBP3O28,SOLVAY,BE,<NA>,2024-12-31 00:00:00+00:00,2025-04-15 16:16:05.785130+00:00,NaT,2025-04-15 16:16:05.785130+00:00,AGGREGATOR_PROCESSED_TIMESTAMP,<NA>,/549300MMVL80RTBP3O28/2024-12-31/ESEF/BE/1/549...,/549300MMVL80RTBP3O28/2024-12-31/ESEF/BE/1/549...,/549300MMVL80RTBP3O28/2024-12-31/ESEF/BE/1/549...,/api/filings/17771,"{""country"": ""BE"", ""date_added"": ""2025-04-15 12...",GAI_1D54C6F46FE0A91920AB,Solvay SA,BE,LINKED
728,17770,549300MMVL80RTBP3O28,SOLVAY,BE,<NA>,2024-12-31 00:00:00+00:00,2025-04-15 16:15:27.809996+00:00,NaT,2025-04-15 16:15:27.809996+00:00,AGGREGATOR_PROCESSED_TIMESTAMP,<NA>,/549300MMVL80RTBP3O28/2024-12-31/ESEF/BE/0/549...,/549300MMVL80RTBP3O28/2024-12-31/ESEF/BE/0/549...,/549300MMVL80RTBP3O28/2024-12-31/ESEF/BE/0/549...,/api/filings/17770,"{""country"": ""BE"", ""date_added"": ""2025-04-15 12...",GAI_1D54C6F46FE0A91920AB,Solvay SA,BE,LINKED
736,17762,549300QRPSGOJRPUFO80,MELEXIS,BE,<NA>,2024-12-31 00:00:00+00:00,2025-04-15 16:12:15.730119+00:00,NaT,2025-04-15 16:12:15.730119+00:00,AGGREGATOR_PROCESSED_TIMESTAMP,<NA>,/549300QRPSGOJRPUFO80/2024-12-31/ESEF/BE/1/mel...,/549300QRPSGOJRPUFO80/2024-12-31/ESEF/BE/1/mel...,/549300QRPSGOJRPUFO80/2024-12-31/ESEF/BE/1/mel...,/api/filings/17762,"{""country"": ""BE"", ""date_added"": ""2025-04-15 12...",GAI_CC2F304AD8F787FB579B,Melexis NV,BE,LINKED
737,17761,549300QRPSGOJRPUFO80,MELEXIS,BE,<NA>,2025-12-31 00:00:00+00:00,2025-04-15 16:11:41.678279+00:00,NaT,2025-04-15 16:11:41.678279+00:00,AGGREGATOR_PROCESSED_TIMESTAMP,<NA>,/549300QRPSGOJRPUFO80/2025-12-31/ESEF/BE/0/mel...,/549300QRPSGOJRPUFO80/2025-12-31/ESEF/BE/0/mel...,/549300QRPSGOJRPUFO80/2025-12-31/ESEF/BE/0/mel...,/api/filings/17761,"{""country"": ""BE"", ""date_added"": ""2025-04-15 12...",GAI_CC2F304AD8F787FB579B,Melexis NV,BE,LINKED
820,13573,549300MMVL80RTBP3O28,SOLVAY,BE,<NA>,2023-12-31 00:00:00+00:00,2024-04-23 11:45:20.948290+00:00,NaT,2024-04-23 11:45:20.948290+00:00,AGGREGATOR_PROCESSED_TIMESTAMP,<NA>,/549300MMVL80RTBP3O28/2023-12-31/ESEF/BE/2/549...,/549300MMVL80RTBP3O28/2023-12-31/ESEF/BE/2/549...,/549300MMVL80RTBP3O28/2023-12-31/ESEF/BE/2/549...,/api/filings/13573,"{""country"": ""BE"", ""date_added"": ""2024-04-23 11...",GAI_1D54C6F46FE0A91920AB,Solvay SA,BE,LINKED
821,13572,549300MMVL80RTBP3O28,SOLVAY,BE,<NA>,2023-12-31 00:00:00+00:00,2024-04-23 11:44:21.324890+00:00,NaT,2024-04-23 11:

In [8]:
# 8. NORMALISE DIRECT RESOURCE URLS FROM THE API
# ------------------------------------------------

def normalise_repository_url(value):
    """
    Convert a relative filings.xbrl.org path to an absolute URL.
    Preserve missing values.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    if not value:
        return pd.NA

    return urljoin(FILINGS_XBRL_BASE, value)


required_resource_columns = {
    "json_url",
    "package_url",
    "viewer_url",
}

missing_resource_columns = required_resource_columns.difference(
    europe_matched_filings_df.columns
)

if missing_resource_columns:
    raise RuntimeError(
        "The filings API output is missing expected resource columns: "
        f"{sorted(missing_resource_columns)}"
    )


europe_filing_resources_df = (
    europe_matched_filings_df[
        [
            "filing_id",
            "lei",
            "issuer_id",
            "issuer_name",
            "country",
            "reporting_date",
            "available_datetime",
            "availability_basis",
            "json_url",
            "package_url",
            "viewer_url",
        ]
    ]
    .drop_duplicates("filing_id")
    .copy()
)

for source_column, target_column in {
    "json_url": "json_resource_url",
    "package_url": "package_resource_url",
    "viewer_url": "viewer_resource_url",
}.items():
    europe_filing_resources_df[target_column] = (
        europe_filing_resources_df[source_column]
        .map(normalise_repository_url)
    )


europe_filing_resource_coverage_df = pd.DataFrame({
    "resource_type": [
        "json_resource_url",
        "package_resource_url",
        "viewer_resource_url",
    ],
    "filings_with_resource": [
        int(europe_filing_resources_df["json_resource_url"].notna().sum()),
        int(europe_filing_resources_df["package_resource_url"].notna().sum()),
        int(europe_filing_resources_df["viewer_resource_url"].notna().sum()),
    ],
    "resource_share": [
        europe_filing_resources_df["json_resource_url"].notna().mean(),
        europe_filing_resources_df["package_resource_url"].notna().mean(),
        europe_filing_resources_df["viewer_resource_url"].notna().mean(),
    ],
})

print(
    f"Matched filings with direct JSON URL: "
    f"{europe_filing_resources_df['json_resource_url'].notna().mean():.2%}"
)

print(
    f"Matched filings with package URL: "
    f"{europe_filing_resources_df['package_resource_url'].notna().mean():.2%}"
)

display(europe_filing_resource_coverage_df)
display(
    europe_filing_resources_df[
        [
            "filing_id",
            "json_resource_url",
            "package_resource_url",
            "viewer_resource_url",
        ]
    ].head(20)
)


Matched filings with direct JSON URL: 99.31%
Matched filings with package URL: 100.00%


,resource_type,filings_with_resource,resource_share
0,json_resource_url,143,0.993056
1,package_resource_url,144,1.000000
2,viewer_resource_url,144,1.000000


,filing_id,json_resource_url,package_resource_url,viewer_resource_url
718,17780,https://filings.xbrl.org/529900F3AIQECS8ZSV61/...,https://filings.xbrl.org/529900F3AIQECS8ZSV61/...,https://filings.xbrl.org/529900F3AIQECS8ZSV61/...
719,17779,https://filings.xbrl.org/529900F3AIQECS8ZSV61/...,https://filings.xbrl.org/529900F3AIQECS8ZSV61/...,https://filings.xbrl.org/529900F3AIQECS8ZSV61/...
726,17772,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...
727,17771,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...
728,17770,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...
736,17762,https://filings.xbrl.org/549300QRPSGOJRPUFO80/...,https://filings.xbrl.org/549300QRPSGOJRPUFO80/...,https://filings.xbrl.org/549300QRPSGOJRPUFO80/...
737,17761,https://filings.xbrl.org/549300QRPSGOJRPUFO80/...,https://filings.xbrl.org/549300QRPSGOJRPUFO80/...,https://filings.xbrl.org/549300QRPSGOJRPUFO80/...
820,13573,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...
821,13572,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...
822,13571,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,https://filings.xbrl.org/549300MMVL80RTBP3O28/...


In [9]:
# 9. DOWNLOAD AND FLATTEN xBRL-JSON FACTS
# ------------------------------------------------

def safe_filename(value: str) -> str:
    value = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(value),
    )
    return value[:180]


def download_direct_json(
    filing_id: str,
    resource_url: str,
) -> tuple[Optional[dict], dict]:

    cache_path = EUROPE_FACT_CACHE_DIR / (
        safe_filename(filing_id) + ".json"
    )

    if cache_path.exists():
        try:
            with cache_path.open(
                "r",
                encoding="utf-8",
            ) as file:
                payload = json.load(file)

            return payload, {
                "filing_id": filing_id,
                "resource_url": resource_url,
                "status": "CACHE_HIT",
                "http_status": 200,
                "extraction_method": "DIRECT_JSON",
                "cache_path": str(cache_path),
            }

        except Exception:
            cache_path.unlink(missing_ok=True)

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                resource_url,
                timeout=REQUEST_TIMEOUT_SECONDS,
                headers={"Accept": "application/json"},
            )

            response.raise_for_status()

            payload = response.json()

            if not isinstance(payload, dict):
                raise ValueError(
                    "Downloaded JSON payload is not an object."
                )

            with cache_path.open(
                "w",
                encoding="utf-8",
            ) as file:
                json.dump(payload, file)

            time.sleep(REQUEST_INTERVAL_SECONDS)

            return payload, {
                "filing_id": filing_id,
                "resource_url": response.url,
                "status": "DOWNLOADED",
                "http_status": response.status_code,
                "extraction_method": "DIRECT_JSON",
                "cache_path": str(cache_path),
                "attempt": attempt,
            }

        except Exception as exc:
            last_error = repr(exc)
            time.sleep(min(2 ** attempt, 20))

    return None, {
        "filing_id": filing_id,
        "resource_url": resource_url,
        "status": "FAILED",
        "http_status": pd.NA,
        "extraction_method": "DIRECT_JSON",
        "error": last_error,
    }


def extract_json_from_package(
    filing_id: str,
    resource_url: str,
) -> tuple[list[tuple[str, dict]], dict]:

    try:
        response = session.get(
            resource_url,
            timeout=REQUEST_TIMEOUT_SECONDS,
            headers={
                "Accept": (
                    "application/zip,"
                    "application/octet-stream"
                )
            },
        )

        response.raise_for_status()

        payloads = []

        with zipfile.ZipFile(
            BytesIO(response.content)
        ) as archive:

            json_members = [
                member_name
                for member_name in archive.namelist()
                if member_name.lower().endswith(".json")
            ]

            for member_name in json_members:
                try:
                    with archive.open(
                        member_name
                    ) as member:
                        payload = json.load(member)

                    if (
                        isinstance(payload, dict)
                        and isinstance(
                            payload.get("facts"),
                            dict,
                        )
                    ):
                        payloads.append(
                            (member_name, payload)
                        )

                except Exception:
                    continue

        time.sleep(REQUEST_INTERVAL_SECONDS)

        return payloads, {
            "filing_id": filing_id,
            "resource_url": resource_url,
            "status": (
                "JSON_EXTRACTED"
                if payloads
                else "NO_XBRL_JSON_IN_PACKAGE"
            ),
            "http_status": response.status_code,
            "extraction_method": "REPORT_PACKAGE",
            "json_reports_found": len(payloads),
        }

    except Exception as exc:
        return [], {
            "filing_id": filing_id,
            "resource_url": resource_url,
            "status": "FAILED",
            "http_status": pd.NA,
            "extraction_method": "REPORT_PACKAGE",
            "error": repr(exc),
        }


def flatten_xbrl_json_facts(
    payload: dict,
    filing_id: str,
    source_json_name: Optional[str] = None,
) -> pd.DataFrame:

    facts = payload.get("facts", {})

    if not isinstance(facts, dict):
        return pd.DataFrame()

    rows = []

    for fact_id, fact in facts.items():

        if not isinstance(fact, dict):
            continue

        dimensions = fact.get(
            "dimensions",
            {},
        ) or {}

        rows.append({
            "filing_id": filing_id,
            "source_json_name": source_json_name,
            "fact_id": fact_id,
            "concept": dimensions.get(
                "concept",
                fact.get("concept"),
            ),
            "entity": dimensions.get(
                "entity",
                fact.get("entity"),
            ),
            "period": dimensions.get(
                "period",
                fact.get("period"),
            ),
            "unit": dimensions.get(
                "unit",
                fact.get("unit"),
            ),
            "language": dimensions.get(
                "language",
                fact.get("language"),
            ),
            "value": fact.get("value"),
            "decimals": fact.get("decimals"),
            "dimensions_json": json.dumps(
                dimensions,
                default=str,
                sort_keys=True,
            ),
            "raw_fact_json": json.dumps(
                fact,
                default=str,
                sort_keys=True,
            ),
        })

    return pd.DataFrame(rows)


fact_frames = []
download_logs = []

download_candidates = europe_filing_resources_df.copy()

if MAX_FILINGS_TO_DOWNLOAD is not None:
    download_candidates = download_candidates.head(
        int(MAX_FILINGS_TO_DOWNLOAD)
    )


if DOWNLOAD_XBRL_JSON:

    for row in tqdm(
        download_candidates.itertuples(index=False),
        total=len(download_candidates),
        desc="Downloading European filing facts",
    ):

        direct_payload = None

        if (
            isinstance(row.json_resource_url, str)
            and row.json_resource_url.startswith("http")
        ):

            direct_payload, direct_log = download_direct_json(
                str(row.filing_id),
                str(row.json_resource_url),
            )

            download_logs.append(direct_log)

            if direct_payload is not None:

                flattened = flatten_xbrl_json_facts(
                    direct_payload,
                    str(row.filing_id),
                    source_json_name=Path(
                        str(row.json_resource_url)
                    ).name,
                )

                if not flattened.empty:
                    fact_frames.append(flattened)

        if (
            direct_payload is None
            and isinstance(
                row.package_resource_url,
                str,
            )
            and row.package_resource_url.startswith("http")
        ):

            package_payloads, package_log = (
                extract_json_from_package(
                    str(row.filing_id),
                    str(row.package_resource_url),
                )
            )

            download_logs.append(package_log)

            for member_name, payload in package_payloads:

                flattened = flatten_xbrl_json_facts(
                    payload,
                    str(row.filing_id),
                    source_json_name=member_name,
                )

                if not flattened.empty:
                    fact_frames.append(flattened)


europe_xbrl_facts_raw_df = (
    pd.concat(
        fact_frames,
        ignore_index=True,
    )
    if fact_frames
    else pd.DataFrame()
)

europe_xbrl_download_log_df = pd.DataFrame(
    download_logs
)

print(
    f"Raw European facts: "
    f"{len(europe_xbrl_facts_raw_df):,}"
)

if not europe_xbrl_download_log_df.empty:
    display(
        europe_xbrl_download_log_df[
            [
                "status",
                "extraction_method",
                "http_status",
            ]
        ]
        .value_counts(dropna=False)
        .rename("count")
        .reset_index()
    )

display(europe_xbrl_download_log_df.head(30))


Raw European facts: 76,683


,status,extraction_method,http_status,count
0,CACHE_HIT,DIRECT_JSON,200,143
1,NO_XBRL_JSON_IN_PACKAGE,REPORT_PACKAGE,200,1


,filing_id,resource_url,status,http_status,extraction_method,cache_path,json_reports_found
0,17780,https://filings.xbrl.org/529900F3AIQECS8ZSV61/...,CACHE_HIT,200,DIRECT_JSON,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,NaN
1,17779,https://filings.xbrl.org/529900F3AIQECS8ZSV61/...,CACHE_HIT,200,DIRECT_JSON,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,NaN
2,17772,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,CACHE_HIT,200,DIRECT_JSON,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,NaN
3,17771,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,CACHE_HIT,200,DIRECT_JSON,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,NaN
4,17770,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,CACHE_HIT,200,DIRECT_JSON,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,NaN
5,17762,https://filings.xbrl.org/549300QRPSGOJRPUFO80/...,CACHE_HIT,200,DIRECT_JSON,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,NaN
6,17761,https://filings.xbrl.org/549300QRPSGOJRPUFO80/...,CACHE_HIT,200,DIRECT_JSON,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,NaN
7,13573,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,CACHE_HIT,200,DIRECT_JSON,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,NaN
8,13572,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,CACHE_HIT,200,DIRECT_JSON,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,NaN
9,13571,https://filings.xbrl.org/549300MMVL80RTBP3O28/...,CACHE_HIT,200,DIRECT_JSON,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,NaN


In [10]:
# 10. PARSE PERIODS, VALUES AND POINT-IN-TIME FILING METADATA
# ------------------------------------------------

def concept_local_name(value):
    if pd.isna(value):
        return pd.NA

    value = str(value)

    if ":" in value:
        value = value.split(":", 1)[1]

    if "}" in value:
        value = value.rsplit("}", 1)[-1]

    return value


def parse_xbrl_period(value):
    if pd.isna(value):
        return pd.NaT, pd.NaT, pd.NA

    text = str(value)

    if "/" in text:
        start_text, end_text = text.split("/", 1)
        return (
            pd.to_datetime(start_text, errors="coerce"),
            pd.to_datetime(end_text, errors="coerce"),
            "DURATION",
        )

    instant = pd.to_datetime(text, errors="coerce")
    return instant, instant, "INSTANT"


if europe_xbrl_facts_raw_df.empty:
    europe_xbrl_facts_pit_df = europe_xbrl_facts_raw_df.copy()
else:
    facts = europe_xbrl_facts_raw_df.copy()

    facts["concept_local_name"] = facts["concept"].map(
        concept_local_name
    )

    parsed_periods = facts["period"].map(parse_xbrl_period)
    facts["period_start"] = parsed_periods.map(lambda item: item[0])
    facts["period_end"] = parsed_periods.map(lambda item: item[1])
    facts["period_type"] = parsed_periods.map(lambda item: item[2])

    facts["numeric_value"] = pd.to_numeric(
        facts["value"],
        errors="coerce",
    )

    filing_link = europe_matched_filings_df[
        [
            "filing_id",
            "lei",
            "issuer_id",
            "issuer_name",
            "country",
            "filing_system",
            "reporting_date",
            "filing_date",
            "processed_datetime",
            "available_datetime",
            "availability_basis",
        ]
    ].drop_duplicates()

    facts = facts.merge(
        filing_link,
        on="filing_id",
        how="left",
        validate="m:m",
    )

    europe_xbrl_facts_pit_df = facts

print(f"Point-in-time European facts: {len(europe_xbrl_facts_pit_df):,}")

Point-in-time European facts: 76,683


In [11]:
# 11. EXPANDED CANONICAL IFRS FUNDAMENTALS
# ------------------------------------------------

# Regional parsers should collect reported source facts. Ratios and derived
# measures such as free cash flow, net debt, margins and returns are intentionally
# deferred to the later global fundamentals layer.


CANONICAL_IFRS_MAPPINGS = [
    # ------------------------------------------------
    # INCOME STATEMENT — CORE
    # ------------------------------------------------
    ("revenue", "Revenue", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("revenue", "RevenueFromContractsWithCustomers", 2, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("revenue", "SalesRevenue", 3, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("cost_of_revenue", "CostOfSales", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("cost_of_revenue", "CostOfRevenue", 2, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("gross_profit", "GrossProfit", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("operating_income", "ProfitLossFromOperatingActivities", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("operating_income", "OperatingProfitLoss", 2, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("profit_before_tax", "ProfitLossBeforeTax", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("profit_before_tax", "ProfitLossFromContinuingOperationsBeforeTax", 2, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("income_tax_expense", "IncomeTaxExpenseContinuingOperations", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("income_tax_expense", "IncomeTaxExpense", 2, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("net_income", "ProfitLoss", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("net_income_attributable_to_owners", "ProfitLossAttributableToOwnersOfParent", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("net_income_attributable_to_nci", "ProfitLossAttributableToNoncontrollingInterests", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),

    ("basic_eps", "BasicEarningsLossPerShare", 1, "INCOME_STATEMENT", "DURATION", "PER_SHARE", 1),
    ("diluted_eps", "DilutedEarningsLossPerShare", 1, "INCOME_STATEMENT", "DURATION", "PER_SHARE", 1),

    ("basic_weighted_average_shares", "WeightedAverageNumberOfSharesOutstandingBasic", 1, "INCOME_STATEMENT", "DURATION", "SHARES", 1),
    ("diluted_weighted_average_shares", "AdjustedWeightedAverageShares", 1, "INCOME_STATEMENT", "DURATION", "SHARES", 1),
    ("diluted_weighted_average_shares", "WeightedAverageNumberOfDilutedSharesOutstanding", 2, "INCOME_STATEMENT", "DURATION", "SHARES", 1),

    # ------------------------------------------------
    # INCOME STATEMENT — OPERATING DETAIL
    # ------------------------------------------------
    ("research_and_development_expense", "ResearchAndDevelopmentExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("selling_expense", "SellingExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("general_and_administrative_expense", "GeneralAndAdministrativeExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("selling_general_and_administrative_expense", "SellingGeneralAndAdministrativeExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("employee_benefit_expense", "EmployeeBenefitsExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("depreciation_expense", "DepreciationExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("amortisation_expense", "AmortisationExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("depreciation_and_amortisation", "DepreciationAndAmortisationExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("impairment_loss", "ImpairmentLoss", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("restructuring_expense", "RestructuringCosts", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 3),
    ("finance_income", "FinanceIncome", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("finance_costs", "FinanceCosts", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("interest_expense", "InterestExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("interest_income", "InterestIncome", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("share_of_profit_equity_method", "ShareOfProfitLossOfAssociatesAndJointVenturesAccountedForUsingEquityMethod", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),

    # ------------------------------------------------
    # COMPREHENSIVE INCOME
    # ------------------------------------------------
    ("other_comprehensive_income", "OtherComprehensiveIncome", 1, "COMPREHENSIVE_INCOME", "DURATION", "MONETARY", 2),
    ("comprehensive_income", "ComprehensiveIncome", 1, "COMPREHENSIVE_INCOME", "DURATION", "MONETARY", 2),
    ("comprehensive_income_attributable_to_owners", "ComprehensiveIncomeAttributableToOwnersOfParent", 1, "COMPREHENSIVE_INCOME", "DURATION", "MONETARY", 2),

    # ------------------------------------------------
    # BALANCE SHEET — ASSETS
    # ------------------------------------------------
    ("total_assets", "Assets", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("current_assets", "CurrentAssets", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("noncurrent_assets", "NoncurrentAssets", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("cash_and_cash_equivalents", "CashAndCashEquivalents", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("cash_and_cash_equivalents", "CashAndCashEquivalentsAtCarryingValue", 2, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("restricted_cash", "RestrictedCash", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("short_term_investments", "OtherCurrentFinancialAssets", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("trade_receivables", "TradeReceivables", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("other_receivables", "OtherReceivables", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("finance_receivables", "FinanceLeaseReceivables", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),

    ("inventory", "Inventories", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("raw_material_inventory", "RawMaterialsAndSupplies", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("work_in_progress_inventory", "WorkInProgress", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("finished_goods_inventory", "FinishedGoods", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),

    ("property_plant_equipment", "PropertyPlantAndEquipment", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("right_of_use_assets", "RightofuseAssets", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("goodwill", "Goodwill", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("intangible_assets", "IntangibleAssetsOtherThanGoodwill", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("capitalised_development_costs", "DevelopmentCosts", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("investments_in_associates", "InvestmentsInAssociates", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("deferred_tax_assets", "DeferredTaxAssets", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("pension_assets", "NetDefinedBenefitAsset", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),

    # ------------------------------------------------
    # BALANCE SHEET — LIABILITIES AND EQUITY
    # ------------------------------------------------
    ("total_liabilities", "Liabilities", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("current_liabilities", "CurrentLiabilities", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("noncurrent_liabilities", "NoncurrentLiabilities", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("trade_payables", "TradePayables", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("other_payables", "OtherPayables", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("contract_liabilities", "ContractLiabilities", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("short_term_debt", "CurrentBorrowings", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("short_term_debt", "ShorttermBorrowings", 2, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("long_term_debt", "NoncurrentBorrowings", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("long_term_debt", "LongtermBorrowings", 2, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("total_borrowings", "Borrowings", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("current_lease_liabilities", "CurrentLeaseLiabilities", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("noncurrent_lease_liabilities", "NoncurrentLeaseLiabilities", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("provisions_current", "CurrentProvisions", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("provisions_noncurrent", "NoncurrentProvisions", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("warranty_provisions", "WarrantyProvisions", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("restructuring_provisions", "RestructuringProvisions", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("pension_liabilities", "NetDefinedBenefitLiability", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("deferred_tax_liabilities", "DeferredTaxLiabilities", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("total_equity", "Equity", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("equity_attributable_to_owners", "EquityAttributableToOwnersOfParent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("noncontrolling_interests", "NoncontrollingInterests", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("share_capital", "IssuedCapital", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("share_premium", "SharePremium", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("retained_earnings", "RetainedEarnings", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("treasury_shares", "TreasuryShares", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("shares_outstanding", "NumberOfSharesOutstanding", 1, "BALANCE_SHEET", "INSTANT", "SHARES", 1),

    # ------------------------------------------------
    # CASH FLOW STATEMENT
    # ------------------------------------------------
    ("operating_cash_flow", "CashFlowsFromUsedInOperatingActivities", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),
    ("investing_cash_flow", "CashFlowsFromUsedInInvestingActivities", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),
    ("financing_cash_flow", "CashFlowsFromUsedInFinancingActivities", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),

    ("capital_expenditure", "PurchaseOfPropertyPlantAndEquipment", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),
    ("capital_expenditure", "PaymentsToAcquirePropertyPlantAndEquipment", 2, "CASH_FLOW", "DURATION", "MONETARY", 1),
    ("intangible_asset_purchases", "PurchaseOfIntangibleAssets", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("business_acquisition_cash_outflow", "CashFlowsUsedInObtainingControlOfSubsidiariesOrOtherBusinesses", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("business_disposal_cash_inflow", "CashFlowsFromLosingControlOfSubsidiariesOrOtherBusinesses", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),

    ("debt_issuance", "ProceedsFromBorrowings", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("debt_repayment", "RepaymentsOfBorrowings", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("lease_payments", "PaymentsOfLeaseLiabilities", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),

    ("dividends_paid", "DividendsPaid", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),
    ("share_repurchases", "PaymentsToAcquireOrRedeemEntitysShares", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("share_issuance_proceeds", "ProceedsFromIssuingShares", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),

    ("interest_paid", "InterestPaid", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("interest_received", "InterestReceived", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("income_taxes_paid", "IncomeTaxesPaid", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),

    ("cash_change", "IncreaseDecreaseInCashAndCashEquivalents", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),

    # ------------------------------------------------
    # AUTOMOTIVE / INDUSTRIAL OPTIONAL DISCLOSURES
    # ------------------------------------------------
    ("vehicle_sales_volume", "VehicleSalesVolume", 1, "OPERATING_METRIC", "DURATION", "COUNT", 3),
    ("vehicle_production_volume", "VehicleProductionVolume", 1, "OPERATING_METRIC", "DURATION", "COUNT", 3),
    ("automotive_revenue", "AutomotiveRevenue", 1, "SEGMENT", "DURATION", "MONETARY", 3),
    ("financial_services_revenue", "FinancialServicesRevenue", 1, "SEGMENT", "DURATION", "MONETARY", 3),
    ("automotive_debt", "AutomotiveDebt", 1, "SEGMENT", "INSTANT", "MONETARY", 3),
    ("financial_services_debt", "FinancialServicesDebt", 1, "SEGMENT", "INSTANT", "MONETARY", 3),
]


europe_standard_concept_dictionary_df = pd.DataFrame(
    CANONICAL_IFRS_MAPPINGS,
    columns=[
        "standard_concept",
        "concept_local_name",
        "priority",
        "statement_type",
        "expected_period_type",
        "expected_unit_family",
        "core_tier",
    ],
)

europe_standard_concept_dictionary_df[
    "taxonomy_family"
] = "IFRS_ESEF"

europe_standard_concept_dictionary_df[
    "is_core"
] = (
    europe_standard_concept_dictionary_df["core_tier"] <= 2
)

europe_standard_concept_dictionary_df[
    "aggregation_policy"
] = np.where(
    europe_standard_concept_dictionary_df[
        "expected_period_type"
    ] == "INSTANT",
    "LATEST_INSTANT",
    "PERIOD_VALUE",
)


def classify_unit_family(unit_value) -> str:
    if pd.isna(unit_value):
        return "UNKNOWN"

    text = str(unit_value).lower()

    if (
        "share" in text
        and (
            "/" not in text
            and "per" not in text
        )
    ):
        return "SHARES"

    if (
        "share" in text
        and (
            "/" in text
            or "per" in text
        )
    ):
        return "PER_SHARE"

    if any(
        token in text
        for token in [
            "eur",
            "usd",
            "gbp",
            "jpy",
            "cny",
            "chf",
            "sek",
            "nok",
            "dkk",
            "pln",
            "huf",
        ]
    ):
        return "MONETARY"

    if any(
        token in text
        for token in [
            "pure",
            "number",
            "count",
            "vehicle",
            "unit",
        ]
    ):
        return "COUNT"

    if "%" in text or "percent" in text:
        return "PERCENTAGE"

    return "OTHER"


if europe_xbrl_facts_pit_df.empty:

    europe_fundamentals_mapped_df = pd.DataFrame()
    europe_fundamentals_standardised_df = pd.DataFrame()
    europe_unmapped_concept_inventory_df = pd.DataFrame()
    europe_automotive_extension_candidates_df = pd.DataFrame()

else:

    facts_for_mapping = europe_xbrl_facts_pit_df.copy()

    facts_for_mapping["observed_unit_family"] = (
        facts_for_mapping["unit"]
        .map(classify_unit_family)
    )

    mapped = facts_for_mapping.merge(
        europe_standard_concept_dictionary_df,
        on="concept_local_name",
        how="left",
        validate="m:m",
    )

    # Preserve every matched candidate before selecting one preferred fact.
    europe_fundamentals_mapped_df = mapped[
        mapped["standard_concept"].notna()
    ].copy()

    europe_fundamentals_mapped_df[
        "period_type_match"
    ] = (
        europe_fundamentals_mapped_df["period_type"]
        == europe_fundamentals_mapped_df[
            "expected_period_type"
        ]
    )

    europe_fundamentals_mapped_df[
        "unit_family_match"
    ] = (
        europe_fundamentals_mapped_df[
            "observed_unit_family"
        ]
        == europe_fundamentals_mapped_df[
            "expected_unit_family"
        ]
    )

    europe_fundamentals_mapped_df[
        "is_numeric_fact"
    ] = europe_fundamentals_mapped_df[
        "numeric_value"
    ].notna()

    europe_fundamentals_mapped_df[
        "is_ifrs_taxonomy_concept"
    ] = (
        europe_fundamentals_mapped_df["concept"]
        .astype("string")
        .str.lower()
        .str.contains("ifrs")
    )

    # Lower scores are preferred. Priority remains dominant, followed by
    # period/unit consistency, numeric availability and core taxonomy status.
    europe_fundamentals_mapped_df[
        "selection_score"
    ] = (
        europe_fundamentals_mapped_df["priority"] * 100
        + (~europe_fundamentals_mapped_df["period_type_match"]) * 20
        + (~europe_fundamentals_mapped_df["unit_family_match"]) * 10
        + (~europe_fundamentals_mapped_df["is_numeric_fact"]) * 5
        + (~europe_fundamentals_mapped_df["is_ifrs_taxonomy_concept"]) * 2
    )

    duplicate_key = [
        "issuer_id",
        "standard_concept",
        "period_start",
        "period_end",
        "available_datetime",
        "unit",
    ]

    duplicate_key = [
        column
        for column in duplicate_key
        if column in europe_fundamentals_mapped_df.columns
    ]

    europe_fundamentals_standardised_df = (
        europe_fundamentals_mapped_df
        .sort_values(
            duplicate_key
            + [
                "selection_score",
                "fact_id",
            ]
        )
        .drop_duplicates(
            subset=duplicate_key,
            keep="first",
        )
        .reset_index(drop=True)
    )

    # Long-form inventory of source concepts not yet represented in the
    # canonical schema. This supports later controlled dictionary expansion.
    unmapped = mapped[
        mapped["standard_concept"].isna()
    ].copy()

    europe_unmapped_concept_inventory_df = (
        unmapped
        .groupby(
            [
                "concept",
                "concept_local_name",
                "period_type",
                "observed_unit_family",
            ],
            dropna=False,
        )
        .agg(
            fact_rows=("fact_id", "size"),
            issuer_count=("issuer_id", "nunique"),
            filing_count=("filing_id", "nunique"),
            numeric_fact_share=(
                "numeric_value",
                lambda series: series.notna().mean(),
            ),
            earliest_available=(
                "available_datetime",
                "min",
            ),
            latest_available=(
                "available_datetime",
                "max",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "issuer_count",
                "fact_rows",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    automotive_pattern = re.compile(
        r"(vehicle|automotive|production|deliver|warranty|"
        r"dealer|finance.?receivable|battery|electric|"
        r"development.?cost|restructur|geograph|segment)",
        flags=re.IGNORECASE,
    )

    europe_automotive_extension_candidates_df = (
        europe_unmapped_concept_inventory_df[
            europe_unmapped_concept_inventory_df[
                "concept_local_name"
            ]
            .astype("string")
            .str.contains(
                automotive_pattern,
                na=False,
            )
        ]
        .copy()
        .reset_index(drop=True)
    )


print(
    "Canonical standard concepts:",
    europe_standard_concept_dictionary_df[
        "standard_concept"
    ].nunique(),
)

print(
    "Source-concept mapping rows:",
    len(europe_standard_concept_dictionary_df),
)

print(
    "Mapped candidate facts:",
    f"{len(europe_fundamentals_mapped_df):,}",
)

print(
    "Selected standardised facts:",
    f"{len(europe_fundamentals_standardised_df):,}",
)

print(
    "Unmapped source concepts inventoried:",
    f"{len(europe_unmapped_concept_inventory_df):,}",
)

print(
    "Automotive extension candidates:",
    f"{len(europe_automotive_extension_candidates_df):,}",
)

display(
    europe_standard_concept_dictionary_df.head(40)
)


Canonical standard concepts: 100
Source-concept mapping rows: 111
Mapped candidate facts: 25,481
Selected standardised facts: 11,756
Unmapped source concepts inventoried: 1,814
Automotive extension candidates: 36


/tmp/ipykernel_989/2407875382.py:431: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


,standard_concept,concept_local_name,priority,statement_type,expected_period_type,expected_unit_family,core_tier,taxonomy_family,is_core,aggregation_policy
0,revenue,Revenue,1,INCOME_STATEMENT,DURATION,MONETARY,1,IFRS_ESEF,True,PERIOD_VALUE
1,revenue,RevenueFromContractsWithCustomers,2,INCOME_STATEMENT,DURATION,MONETARY,1,IFRS_ESEF,True,PERIOD_VALUE
2,revenue,SalesRevenue,3,INCOME_STATEMENT,DURATION,MONETARY,1,IFRS_ESEF,True,PERIOD_VALUE
3,cost_of_revenue,CostOfSales,1,INCOME_STATEMENT,DURATION,MONETARY,1,IFRS_ESEF,True,PERIOD_VALUE
4,cost_of_revenue,CostOfRevenue,2,INCOME_STATEMENT,DURATION,MONETARY,1,IFRS_ESEF,True,PERIOD_VALUE
5,gross_profit,GrossProfit,1,INCOME_STATEMENT,DURATION,MONETARY,1,IFRS_ESEF,True,PERIOD_VALUE
6,operating_income,ProfitLossFromOperatingActivities,1,INCOME_STATEMENT,DURATION,MONETARY,1,IFRS_ESEF,True,PERIOD_VALUE
7,operating_income,OperatingProfitLoss,2,INCOME_STATEMENT,DURATION,MONETARY,1,IFRS_ESEF,True,PERIOD_VALUE
8,profit_before_tax,ProfitLossBeforeTax,1,INCOME_STATEMENT,DURATION,MONETARY,1,IFRS_ESEF,True,PERIOD_VALUE
9,profit_before_tax,ProfitLossFromContinuingOperationsBeforeTax,2,INCOME_STATEMENT,DURATION,MONETARY,1,IFRS_ESEF,True,PERIOD_VALUE


In [12]:
# 12. ATTACH LISTED SECURITY IDs
# ------------------------------------------------

def attach_security_ids(
    issuer_level_facts: pd.DataFrame,
    security_universe: pd.DataFrame,
) -> pd.DataFrame:

    if issuer_level_facts.empty:
        return issuer_level_facts.copy()

    bridge = (
        security_universe[
            [
                column
                for column in [
                    "issuer_id",
                    "security_id",
                    "ticker",
                    "country",
                    "lei",
                ]
                if column
                in security_universe.columns
            ]
        ]
        .dropna(
            subset=[
                "issuer_id",
                "security_id",
            ]
        )
        .drop_duplicates()
    )

    join_columns = [
        column
        for column in [
            "issuer_id",
            "country",
            "lei",
        ]
        if (
            column
            in issuer_level_facts.columns
            and column
            in bridge.columns
        )
    ]

    if "issuer_id" not in join_columns:
        raise KeyError(
            "issuer_id is required for "
            "European security expansion."
        )

    return issuer_level_facts.merge(
        bridge,
        on=join_columns,
        how="left",
        suffixes=(
            "",
            "_security",
        ),
        validate="m:m",
    )


europe_fundamentals_security_linked_df = (
    attach_security_ids(
        europe_fundamentals_standardised_df,
        europe_issuer_security_universe_df,
    )
)

issuer_level_rows = len(
    europe_fundamentals_standardised_df
)

security_expanded_rows = len(
    europe_fundamentals_security_linked_df
)

europe_issuer_security_link_quality_df = (
    pd.DataFrame({
        "metric": [
            "issuer_level_fact_rows",
            "security_expanded_fact_rows",
            "row_multiplication_ratio",
            "issuer_level_issuers",
            "security_expanded_issuers",
            "linked_security_count",
            "security_linked_row_share",
            "rows_without_security_id",
        ],
        "value": [
            issuer_level_rows,
            security_expanded_rows,
            (
                security_expanded_rows
                / issuer_level_rows
                if issuer_level_rows > 0
                else np.nan
            ),
            (
                europe_fundamentals_standardised_df[
                    "issuer_id"
                ].nunique()
                if not europe_fundamentals_standardised_df.empty
                else 0
            ),
            (
                europe_fundamentals_security_linked_df[
                    "issuer_id"
                ].nunique()
                if not europe_fundamentals_security_linked_df.empty
                else 0
            ),
            (
                europe_fundamentals_security_linked_df[
                    "security_id"
                ].nunique()
                if (
                    not europe_fundamentals_security_linked_df.empty
                    and "security_id"
                    in europe_fundamentals_security_linked_df.columns
                )
                else 0
            ),
            (
                europe_fundamentals_security_linked_df[
                    "security_id"
                ].notna().mean()
                if (
                    not europe_fundamentals_security_linked_df.empty
                    and "security_id"
                    in europe_fundamentals_security_linked_df.columns
                )
                else np.nan
            ),
            (
                int(
                    europe_fundamentals_security_linked_df[
                        "security_id"
                    ].isna().sum()
                )
                if (
                    not europe_fundamentals_security_linked_df.empty
                    and "security_id"
                    in europe_fundamentals_security_linked_df.columns
                )
                else 0
            ),
        ],
    })
)

print(
    "Security-linked European fundamentals:",
    f"{security_expanded_rows:,}",
)

display(
    europe_issuer_security_link_quality_df
)

Security-linked European fundamentals: 13,063


,metric,value
0,issuer_level_fact_rows,11756.000000
1,security_expanded_fact_rows,13063.000000
2,row_multiplication_ratio,1.111177
3,issuer_level_issuers,30.000000
4,security_expanded_issuers,30.000000
5,linked_security_count,30.000000
6,security_linked_row_share,0.924367
7,rows_without_security_id,988.000000


In [13]:
# 13. POINT-IN-TIME ACCESS FUNCTIONS
# ------------------------------------------------

def european_fundamentals_as_of(
    dataframe: pd.DataFrame,
    as_of_date,
    *,
    issuer_ids: Optional[Iterable[str]] = None,
    security_ids: Optional[Iterable[str]] = None,
    standard_concepts: Optional[Iterable[str]] = None,
) -> pd.DataFrame:

    if dataframe.empty:
        return dataframe.copy()

    cutoff = pd.Timestamp(as_of_date)

    if cutoff.tzinfo is None:
        cutoff = cutoff.tz_localize("UTC")
    else:
        cutoff = cutoff.tz_convert("UTC")

    result = dataframe[
        pd.to_datetime(
            dataframe["available_datetime"],
            errors="coerce",
            utc=True,
        ) <= cutoff
    ].copy()

    if issuer_ids is not None:
        result = result[
            result["issuer_id"].isin(set(issuer_ids))
        ]

    if (
        security_ids is not None
        and "security_id" in result.columns
    ):
        result = result[
            result["security_id"].isin(set(security_ids))
        ]

    if standard_concepts is not None:
        result = result[
            result["standard_concept"].isin(
                set(standard_concepts)
            )
        ]

    return result


def latest_european_fact_as_of(
    dataframe: pd.DataFrame,
    as_of_date,
) -> pd.DataFrame:

    result = european_fundamentals_as_of(
        dataframe,
        as_of_date,
    )

    if result.empty:
        return result

    grouping = [
        column
        for column in [
            "security_id",
            "issuer_id",
            "standard_concept",
        ]
        if column in result.columns
    ]

    return (
        result
        .sort_values(
            ["period_end", "available_datetime"]
        )
        .drop_duplicates(
            grouping,
            keep="last",
        )
        .reset_index(drop=True)
    )

In [14]:
# 14. COVERAGE AND QUALITY-CONTROL REPORTS
# ------------------------------------------------

issuer_coverage = (
    europe_issuer_universe_df
    .groupby("country", dropna=False)
    .agg(
        universe_issuers=("issuer_id", "nunique"),
        issuers_with_lei=("lei", lambda series: series.notna().sum()),
    )
    .reset_index()
)

filing_coverage = (
    europe_matched_filings_df
    .groupby("country", dropna=False)
    .agg(
        matched_issuers=("issuer_id", "nunique"),
        matched_filings=("filing_id", "nunique"),
        earliest_report=("reporting_date", "min"),
        latest_report=("reporting_date", "max"),
    )
    .reset_index()
    if not europe_matched_filings_df.empty
    else pd.DataFrame(
        columns=[
            "country",
            "matched_issuers",
            "matched_filings",
            "earliest_report",
            "latest_report",
        ]
    )
)

europe_coverage_report_df = (
    issuer_coverage
    .merge(
        filing_coverage,
        on="country",
        how="left",
    )
    .merge(
        europe_filing_system_registry_df[
            [
                "country_code",
                "system_name",
                "connector_status",
            ]
        ],
        left_on="country",
        right_on="country_code",
        how="left",
    )
)

for column in ["matched_issuers", "matched_filings"]:
    europe_coverage_report_df[column] = (
        europe_coverage_report_df[column]
        .fillna(0)
        .astype(int)
    )

europe_coverage_report_df["issuer_match_rate"] = np.where(
    europe_coverage_report_df["universe_issuers"] > 0,
    europe_coverage_report_df["matched_issuers"]
    / europe_coverage_report_df["universe_issuers"],
    np.nan,
)

europe_issuer_mapping_gaps_df = (
    europe_issuer_universe_df[
        ~europe_issuer_universe_df["issuer_id"].isin(
            set(
                europe_matched_filings_df["issuer_id"].dropna()
            )
        )
    ]
    .copy()
)

europe_fact_quality_df = pd.DataFrame({
    "metric": [
        "raw_fact_rows",
        "standardised_fact_rows",
        "numeric_raw_fact_share",
        "facts_with_available_datetime_share",
        "security_link_rate",
        "failed_json_downloads",
    ],
    "value": [
        len(europe_xbrl_facts_raw_df),
        len(europe_fundamentals_standardised_df),
        (
            europe_xbrl_facts_pit_df["numeric_value"].notna().mean()
            if not europe_xbrl_facts_pit_df.empty
            else np.nan
        ),
        (
            europe_xbrl_facts_pit_df["available_datetime"].notna().mean()
            if not europe_xbrl_facts_pit_df.empty
            else np.nan
        ),
        (
            europe_fundamentals_security_linked_df["security_id"].notna().mean()
            if (
                not europe_fundamentals_security_linked_df.empty
                and "security_id" in europe_fundamentals_security_linked_df.columns
            )
            else np.nan
        ),
        (
            int(
                (
                    europe_xbrl_download_log_df["status"] == "FAILED"
                ).sum()
            )
            if not europe_xbrl_download_log_df.empty
            else 0
        ),
    ],
})

display(europe_coverage_report_df)
display(europe_fact_quality_df)

# Expanded canonical-schema diagnostics.
europe_standard_concept_coverage_df = (
    europe_fundamentals_standardised_df
    .groupby(
        [
            "standard_concept",
            "statement_type",
            "core_tier",
        ],
        dropna=False,
    )
    .agg(
        fact_rows=("fact_id", "size"),
        issuer_count=("issuer_id", "nunique"),
        filing_count=("filing_id", "nunique"),
        earliest_period=("period_end", "min"),
        latest_period=("period_end", "max"),
        numeric_fact_share=(
            "numeric_value",
            lambda series: series.notna().mean(),
        ),
        period_match_share=(
            "period_type_match",
            "mean",
        ),
        unit_match_share=(
            "unit_family_match",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "core_tier",
            "issuer_count",
            "fact_rows",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
    if not europe_fundamentals_standardised_df.empty
    else pd.DataFrame()
)

europe_mapping_quality_df = pd.DataFrame({
    "metric": [
        "canonical_standard_concepts",
        "source_mapping_rows",
        "mapped_candidate_fact_rows",
        "selected_standardised_fact_rows",
        "unique_standard_concepts_observed",
        "period_type_match_share",
        "unit_family_match_share",
        "unmapped_concepts_in_inventory",
        "automotive_extension_candidates",
    ],
    "value": [
        europe_standard_concept_dictionary_df[
            "standard_concept"
        ].nunique(),
        len(europe_standard_concept_dictionary_df),
        len(europe_fundamentals_mapped_df),
        len(europe_fundamentals_standardised_df),
        (
            europe_fundamentals_standardised_df[
                "standard_concept"
            ].nunique()
            if not europe_fundamentals_standardised_df.empty
            else 0
        ),
        (
            europe_fundamentals_standardised_df[
                "period_type_match"
            ].mean()
            if not europe_fundamentals_standardised_df.empty
            else np.nan
        ),
        (
            europe_fundamentals_standardised_df[
                "unit_family_match"
            ].mean()
            if not europe_fundamentals_standardised_df.empty
            else np.nan
        ),
        len(europe_unmapped_concept_inventory_df),
        len(europe_automotive_extension_candidates_df),
    ],
})

# ------------------------------------------------
# STANDARD-CONCEPT COVERAGE SUMMARY
# ------------------------------------------------

def build_standard_concept_availability_report(
    standardised_facts: pd.DataFrame,
    canonical_dictionary: pd.DataFrame,
) -> pd.DataFrame:
    """
    Summarise the practical availability of each canonical fundamental.

    The report includes all canonical concepts, including concepts that are
    defined in the dictionary but not observed in the current European universe.
    """

    dictionary_summary = (
        canonical_dictionary[
            [
                "standard_concept",
                "statement_type",
                "expected_period_type",
                "expected_unit_family",
                "core_tier",
                "is_core",
            ]
        ]
        .drop_duplicates("standard_concept")
        .copy()
    )

    if standardised_facts.empty:
        dictionary_summary["issuer_coverage"] = 0
        dictionary_summary["security_coverage"] = 0
        dictionary_summary["filing_coverage"] = 0
        dictionary_summary["fact_rows"] = 0
        dictionary_summary["first_reporting_date"] = pd.NaT
        dictionary_summary["last_reporting_date"] = pd.NaT
        dictionary_summary["first_available_datetime"] = pd.NaT
        dictionary_summary["last_available_datetime"] = pd.NaT
        dictionary_summary["numeric_fact_share"] = np.nan
        dictionary_summary["issuer_coverage_rate"] = 0.0

        return dictionary_summary.sort_values(
            ["core_tier", "statement_type", "standard_concept"]
        ).reset_index(drop=True)

    facts = standardised_facts.copy()

    facts["period_end"] = pd.to_datetime(
        facts["period_end"],
        errors="coerce",
        utc=True,
    )

    facts["available_datetime"] = pd.to_datetime(
        facts["available_datetime"],
        errors="coerce",
        utc=True,
    )

    aggregation_spec = {
        "issuer_coverage": (
            "issuer_id",
            lambda series: series.dropna().nunique(),
        ),
        "filing_coverage": (
            "filing_id",
            lambda series: series.dropna().nunique(),
        ),
        "fact_rows": (
            "fact_id",
            "size",
        ),
        "first_reporting_date": (
            "period_end",
            "min",
        ),
        "last_reporting_date": (
            "period_end",
            "max",
        ),
        "first_available_datetime": (
            "available_datetime",
            "min",
        ),
        "last_available_datetime": (
            "available_datetime",
            "max",
        ),
        "numeric_fact_share": (
            "numeric_value",
            lambda series: series.notna().mean(),
        ),
    }

    if "security_id" in facts.columns:
        aggregation_spec["security_coverage"] = (
            "security_id",
            lambda series: series.dropna().nunique(),
        )

    observed_coverage = (
        facts
        .groupby(
            "standard_concept",
            dropna=False,
        )
        .agg(**aggregation_spec)
        .reset_index()
    )

    if "security_coverage" not in observed_coverage.columns:
        observed_coverage["security_coverage"] = 0

    total_issuers = max(
        int(
            europe_issuer_universe_df["issuer_id"]
            .dropna()
            .nunique()
        ),
        1,
    )

    report = dictionary_summary.merge(
        observed_coverage,
        on="standard_concept",
        how="left",
    )

    count_columns = [
        "issuer_coverage",
        "security_coverage",
        "filing_coverage",
        "fact_rows",
    ]

    for column in count_columns:
        report[column] = (
            report[column]
            .fillna(0)
            .astype(int)
        )

    report["issuer_coverage_rate"] = (
        report["issuer_coverage"] / total_issuers
    )

    report["observed_in_current_universe"] = (
        report["fact_rows"] > 0
    )

    report["coverage_class"] = pd.cut(
        report["issuer_coverage_rate"],
        bins=[
            -0.001,
            0.10,
            0.30,
            0.60,
            0.80,
            1.00,
        ],
        labels=[
            "VERY_SPARSE",
            "SPARSE",
            "MODERATE",
            "HIGH",
            "VERY_HIGH",
        ],
    )

    report["first_reporting_year"] = (
        report["first_reporting_date"]
        .dt.year
        .astype("Int64")
    )

    report["last_reporting_year"] = (
        report["last_reporting_date"]
        .dt.year
        .astype("Int64")
    )

    return (
        report
        .sort_values(
            [
                "core_tier",
                "issuer_coverage_rate",
                "filing_coverage",
                "standard_concept",
            ],
            ascending=[
                True,
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )


europe_standard_concept_availability_df = (
    build_standard_concept_availability_report(
        europe_fundamentals_standardised_df,
        europe_standard_concept_dictionary_df,
    )
)

display(
    europe_standard_concept_availability_df[
        [
            "standard_concept",
            "statement_type",
            "core_tier",
            "issuer_coverage",
            "issuer_coverage_rate",
            "security_coverage",
            "filing_coverage",
            "fact_rows",
            "first_reporting_year",
            "last_reporting_year",
            "numeric_fact_share",
            "coverage_class",
        ]
    ].head(100)
)

display(europe_standard_concept_coverage_df.head(100))
display(europe_mapping_quality_df)


# ------------------------------------------------
# ISSUER-CENTRIC LINKAGE QA
# ------------------------------------------------

europe_lei_link_quality_df = pd.DataFrame({
    "metric": [
        "economic_issuer_rows",
        "economic_issuer_count",
        "issuer_rows_with_lei",
        "confirmed_lei_issuer_bridges",
        "conflicted_lei_rows",
        "filing_rows",
        "filing_rows_with_issuer_id",
        "filing_rows_missing_issuer_id",
        "matched_filing_issuer_count",
        "preferred_accounting_source_rows",
    ],
    "value": [
        len(
            europe_economic_issuer_universe_df
        ),
        europe_economic_issuer_universe_df[
            "issuer_id"
        ].nunique(),
        int(
            europe_economic_issuer_universe_df[
                "lei"
            ].notna().sum()
        ),
        len(
            europe_lei_issuer_bridge_df
        ),
        len(
            europe_lei_issuer_conflicts_df
        ),
        len(
            europe_filing_metadata_df
        ),
        int(
            europe_filing_metadata_df[
                "issuer_id"
            ].notna().sum()
        ),
        int(
            europe_filing_metadata_df[
                "issuer_id"
            ].isna().sum()
        ),
        europe_filing_metadata_df[
            "issuer_id"
        ].nunique(),
        len(
            europe_preferred_accounting_source_df
        ),
    ],
})

display(
    europe_lei_link_quality_df
)


,country,universe_issuers,issuers_with_lei,matched_issuers,matched_filings,earliest_report,latest_report,country_code,system_name,connector_status,issuer_match_rate
0,AT,1,1,0,0,NaT,NaT,AT,OeKB Issuer Information,FEDERATED,0.000000
1,BE,3,3,3,27,2021-12-31 00:00:00+00:00,2025-12-31 00:00:00+00:00,BE,FSMA / STORI,FEDERATED,1.000000
2,CH,5,4,0,0,NaT,NaT,CH,SIX Exchange Regulation / issuer sources,FEDERATED,0.000000
3,DE,19,16,0,0,NaT,NaT,DE,Unternehmensregister,NATIONAL_CONNECTOR_REQUIRED,0.000000
4,ES,2,2,2,9,2020-12-31 00:00:00+00:00,2024-12-31 00:00:00+00:00,ES,CNMV,FEDERATED,1.000000
5,FI,1,1,1,3,2024-12-31 00:00:00+00:00,2025-12-31 00:00:00+00:00,FI,Finanssivalvonta / Nasdaq,FEDERATED,1.000000
6,FR,9,9,6,32,2020-12-31 00:00:00+00:00,2025-12-31 00:00:00+00:00,FR,AMF / INFO-FINANCIERE,FEDERATED,0.666667
7,GB,14,11,7,26,2021-12-31 00:00:00+00:00,2026-03-31 00:00:00+00:00,GB,FCA National Storage Mechanism,FEDERATED,0.500000
8,IE,5,4,0,0,NaT,NaT,IE,Central Bank of Ireland,NATIONAL_CONNECTOR_REQUIRED,0.000000
9,IT,3,3,1,2,2024-12-31 00:00:00+00:00,2025-12-31 00:00:00+00:00,IT,CONSOB / authorised storage systems,FEDERATED,0.333333


,metric,value
0,raw_fact_rows,76683.000000
1,standardised_fact_rows,11756.000000
2,numeric_raw_fact_share,0.804363
3,facts_with_available_datetime_share,1.000000
4,security_link_rate,0.924367
5,failed_json_downloads,0.000000


,standard_concept,statement_type,core_tier,issuer_coverage,issuer_coverage_rate,security_coverage,filing_coverage,fact_rows,first_reporting_year,last_reporting_year,numeric_fact_share,coverage_class
0,cash_and_cash_equivalents,BALANCE_SHEET,1,30,0.405405,0,143,394,2019,2026,1.000000,MODERATE
1,income_tax_expense,INCOME_STATEMENT,1,30,0.405405,0,143,301,2020,2026,1.000000,MODERATE
2,net_income,INCOME_STATEMENT,1,30,0.405405,0,143,304,2020,2026,1.000000,MODERATE
3,total_equity,BALANCE_SHEET,1,30,0.405405,0,143,453,2019,2026,1.000000,MODERATE
4,operating_cash_flow,CASH_FLOW,1,30,0.405405,0,137,289,2020,2026,1.000000,MODERATE
5,current_assets,BALANCE_SHEET,1,30,0.405405,0,136,275,2020,2026,1.000000,MODERATE
6,revenue,INCOME_STATEMENT,1,29,0.391892,0,142,299,2020,2026,1.000000,MODERATE
7,total_assets,BALANCE_SHEET,1,29,0.391892,0,140,282,2020,2026,1.000000,MODERATE
8,operating_income,INCOME_STATEMENT,1,29,0.391892,0,138,291,2020,2026,1.000000,MODERATE
9,financing_cash_flow,CASH_FLOW,1,29,0.391892,0,131,277,2020,2026,1.000000,MODERATE


,standard_concept,statement_type,core_tier,fact_rows,issuer_count,filing_count,earliest_period,latest_period,numeric_fact_share,period_match_share,unit_match_share
64,total_equity,BALANCE_SHEET,1.0,453,30,143,2019-01-01,2026-04-01,1.000000,1.0,1.0
2,cash_and_cash_equivalents,BALANCE_SHEET,1.0,394,30,143,2019-01-01,2026-04-01,1.000000,1.0,1.0
32,net_income,INCOME_STATEMENT,1.0,304,30,143,2020-01-01,2026-04-01,1.000000,1.0,1.0
25,income_tax_expense,INCOME_STATEMENT,1.0,301,30,143,2020-01-01,2026-04-01,1.000000,1.0,1.0
39,operating_cash_flow,CASH_FLOW,1.0,289,30,137,2020-01-01,2026-04-01,1.000000,1.0,1.0
7,current_assets,BALANCE_SHEET,1.0,275,30,136,2020-01-01,2026-04-01,1.000000,1.0,1.0
50,revenue,INCOME_STATEMENT,1.0,299,29,142,2020-01-01,2026-04-01,1.000000,1.0,1.0
40,operating_income,INCOME_STATEMENT,1.0,291,29,138,2020-01-01,2026-04-01,1.000000,1.0,1.0
62,total_assets,BALANCE_SHEET,1.0,282,29,140,2020-01-01,2026-04-01,1.000000,1.0,1.0
20,financing_cash_flow,CASH_FLOW,1.0,277,29,131,2020-01-01,2026-04-01,1.000000,1.0,1.0


,metric,value
0,canonical_standard_concepts,100.0
1,source_mapping_rows,111.0
2,mapped_candidate_fact_rows,25481.0
3,selected_standardised_fact_rows,11756.0
4,unique_standard_concepts_observed,69.0
5,period_type_match_share,1.0
6,unit_family_match_share,1.0
7,unmapped_concepts_in_inventory,1814.0
8,automotive_extension_candidates,36.0


,metric,value
0,economic_issuer_rows,77
1,economic_issuer_count,74
2,issuer_rows_with_lei,67
3,confirmed_lei_issuer_bridges,64
4,conflicted_lei_rows,0
5,filing_rows,10281
6,filing_rows_with_issuer_id,144
7,filing_rows_missing_issuer_id,10137
8,matched_filing_issuer_count,30
9,preferred_accounting_source_rows,64


In [15]:
# 15. BLOCK 4 OUTPUT CONTRACT
# ------------------------------------------------

block_4_data = {
    "europe_filing_system_registry_df": europe_filing_system_registry_df,
    "europe_security_universe_df": europe_security_universe_df,
    "europe_issuer_universe_df": europe_issuer_universe_df,
    "europe_economic_issuer_universe_df": europe_economic_issuer_universe_df,
    "europe_issuer_security_universe_df": europe_issuer_security_universe_df,
    "europe_lei_issuer_bridge_candidates_df": europe_lei_issuer_bridge_candidates_df,
    "europe_lei_issuer_bridge_df": europe_lei_issuer_bridge_df,
    "europe_lei_issuer_conflicts_df": europe_lei_issuer_conflicts_df,
    "europe_preferred_accounting_source_df": europe_preferred_accounting_source_df,
    "europe_entity_relationship_graph_df": europe_entity_relationship_graph_df,
    "europe_filings_discovered_df": europe_filings_discovered_df,
    "europe_filing_discovery_log_df": europe_filing_discovery_log_df,
    "europe_filing_metadata_df": europe_filing_metadata_df,
    "europe_matched_filings_df": europe_matched_filings_df,
    "europe_unmatched_filings_df": europe_unmatched_filings_df,
    "europe_filing_resources_df": europe_filing_resources_df,
    "europe_filing_resource_coverage_df": europe_filing_resource_coverage_df,
    "europe_xbrl_download_log_df": europe_xbrl_download_log_df,
    "europe_xbrl_facts_raw_df": europe_xbrl_facts_raw_df,
    "europe_xbrl_facts_pit_df": europe_xbrl_facts_pit_df,
    "europe_standard_concept_dictionary_df": europe_standard_concept_dictionary_df,
    "europe_fundamentals_mapped_df": europe_fundamentals_mapped_df,
    "europe_fundamentals_standardised_df": europe_fundamentals_standardised_df,
    "europe_fundamentals_security_linked_df": europe_fundamentals_security_linked_df,
    "europe_unmapped_concept_inventory_df": europe_unmapped_concept_inventory_df,
    "europe_automotive_extension_candidates_df": europe_automotive_extension_candidates_df,
    "europe_standard_concept_coverage_df": europe_standard_concept_coverage_df,
    "europe_standard_concept_availability_df": europe_standard_concept_availability_df,
    "europe_mapping_quality_df": europe_mapping_quality_df,
    "europe_coverage_report_df": europe_coverage_report_df,
    "europe_issuer_mapping_gaps_df": europe_issuer_mapping_gaps_df,
    "europe_fact_quality_df": europe_fact_quality_df,
    "europe_lei_link_quality_df": europe_lei_link_quality_df,
    "europe_issuer_security_link_quality_df": europe_issuer_security_link_quality_df,
}

print("Block 4 transformations complete.")

for name in [
    "europe_economic_issuer_universe_df",
    "europe_issuer_security_universe_df",
    "europe_lei_issuer_bridge_df",
    "europe_filing_metadata_df",
    "europe_fundamentals_standardised_df",
    "europe_fundamentals_security_linked_df",
]:
    print(
        f"  {name}: "
        f"{len(block_4_data[name]):,} rows"
    )

Block 4 transformations complete.
  europe_economic_issuer_universe_df: 77 rows
  europe_issuer_security_universe_df: 81 rows
  europe_lei_issuer_bridge_df: 64 rows
  europe_filing_metadata_df: 10,281 rows
  europe_fundamentals_standardised_df: 11,756 rows
  europe_fundamentals_security_linked_df: 13,063 rows


In [16]:
# 16. PERSIST BLOCK 4 OUTPUTS
# ------------------------------------------------

def make_parquet_safe(dataframe: pd.DataFrame) -> pd.DataFrame:

    output = dataframe.copy()

    for column in output.columns:
        if output[column].dtype == "object":
            non_missing = output[column].dropna()

            if (
                not non_missing.empty
                and non_missing.map(type).nunique() > 1
            ):
                output[column] = output[column].astype("string")

    return output


def persist_dataframe(
    name: str,
    dataframe: pd.DataFrame,
    output_dir: Path,
    *,
    overwrite: bool = True,
) -> dict:

    output_path = output_dir / f"{name}.parquet"

    if output_path.exists() and not overwrite:
        raise FileExistsError(
            f"Refusing to overwrite existing output: {output_path}"
        )

    safe_df = make_parquet_safe(dataframe)

    safe_df.to_parquet(
        output_path,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )

    return {
        "table_name": name,
        "path": str(output_path),
        "row_count": int(len(safe_df)),
        "column_count": int(len(safe_df.columns)),
        "columns": list(map(str, safe_df.columns)),
        "file_size_bytes": int(output_path.stat().st_size),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }


def load_block_4_outputs(
    output_dir: Path = BLOCK_4_OUTPUT_DIR,
) -> dict[str, pd.DataFrame]:

    manifest_path = output_dir / "block_4_manifest.json"

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Block 4 manifest not found: {manifest_path}"
        )

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    loaded = {}

    for table in manifest["tables"]:
        table_path = Path(table["path"])

        if not table_path.exists():
            raise FileNotFoundError(
                f"Manifest table is missing: {table_path}"
            )

        loaded[table["table_name"]] = pd.read_parquet(table_path)

    return loaded


if PERSIST_BLOCK_4_OUTPUTS:

    manifest_rows = []

    for table_name, dataframe in block_4_data.items():
        manifest_rows.append(
            persist_dataframe(
                table_name,
                dataframe,
                BLOCK_4_OUTPUT_DIR,
                overwrite=OVERWRITE_PERSISTED_OUTPUTS,
            )
        )

    block_4_manifest = {
        "block": 4,
        "block_name": "European filing architecture: ESEF, UKSEF and national systems",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "project_root": str(PROJECT_ROOT),
        "input_manifests": [
            str(BLOCK_2_MANIFEST_PATH),
            str(BLOCK_3_MANIFEST_PATH),
        ],
        "output_directory": str(BLOCK_4_OUTPUT_DIR),
        "discovery_source": "filings.xbrl.org public JSON-API",
        "official_architecture": "National OAM / NSM systems with ESEF or UKSEF reports",
        "as_of_date": AS_OF_DATE,
        "countries_queried": countries_to_query,
        "tables": manifest_rows,
    }

    with BLOCK_4_MANIFEST_PATH.open("w", encoding="utf-8") as file:
        json.dump(block_4_manifest, file, indent=2)

    block_4_persistence_report_df = pd.DataFrame(manifest_rows)

    print("Block 4 outputs persisted successfully.")
    print("Manifest:", BLOCK_4_MANIFEST_PATH)

    display(
        block_4_persistence_report_df[
            [
                "table_name",
                "row_count",
                "column_count",
                "file_size_bytes",
                "path",
            ]
        ]
    )

else:

    block_4_persistence_report_df = pd.DataFrame()

    print(
        "PERSIST_BLOCK_4_OUTPUTS is False. "
        "Outputs remain available only in the current runtime."
    )

Block 4 outputs persisted successfully.
Manifest: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_4/block_4_manifest.json


,table_name,row_count,column_count,file_size_bytes,path
0,europe_filing_system_registry_df,32,8,6238,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
1,europe_security_universe_df,81,6,10374,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
2,europe_issuer_universe_df,77,4,7207,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
3,europe_economic_issuer_universe_df,77,4,7207,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
4,europe_issuer_security_universe_df,81,6,10374,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
5,europe_lei_issuer_bridge_candidates_df,67,7,8624,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
6,europe_lei_issuer_bridge_df,64,7,8514,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
7,europe_lei_issuer_conflicts_df,0,7,3654,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
8,europe_preferred_accounting_source_df,64,8,9521,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
9,europe_entity_relationship_graph_df,0,8,4236,/content/drive/MyDrive/Colab Notebooks/00 A1 A...


In [17]:
# 17. PERSISTENCE VALIDATION
# ------------------------------------------------

if PERSIST_BLOCK_4_OUTPUTS:

    reloaded_block_4_data = (
        load_block_4_outputs()
    )

    required_downstream_tables = {
        "europe_filing_system_registry_df",
        "europe_economic_issuer_universe_df",
        "europe_issuer_security_universe_df",
        "europe_lei_issuer_bridge_df",
        "europe_lei_issuer_conflicts_df",
        "europe_preferred_accounting_source_df",
        "europe_entity_relationship_graph_df",
        "europe_filing_metadata_df",
        "europe_fundamentals_standardised_df",
        "europe_fundamentals_security_linked_df",
        "europe_standard_concept_dictionary_df",
        "europe_standard_concept_coverage_df",
        "europe_standard_concept_availability_df",
        "europe_unmapped_concept_inventory_df",
        "europe_coverage_report_df",
        "europe_lei_link_quality_df",
        "europe_issuer_security_link_quality_df",
    }

    missing = (
        required_downstream_tables
        .difference(
            reloaded_block_4_data
        )
    )

    if missing:
        raise RuntimeError(
            "Persistence validation failed. "
            f"Missing tables: {sorted(missing)}"
        )

    validation_rows = []

    for table_name in sorted(
        required_downstream_tables
    ):
        original_rows = len(
            block_4_data[
                table_name
            ]
        )

        reloaded_rows = len(
            reloaded_block_4_data[
                table_name
            ]
        )

        if original_rows != reloaded_rows:
            raise RuntimeError(
                f"Row-count mismatch for "
                f"{table_name}: "
                f"{original_rows} original versus "
                f"{reloaded_rows} reloaded."
            )

        validation_rows.append({
            "table_name": table_name,
            "expected_rows": original_rows,
            "persisted_rows": reloaded_rows,
            "status": "PASSED",
        })

    reloaded_facts_df = (
        reloaded_block_4_data[
            "europe_fundamentals_standardised_df"
        ]
    )

    if (
        len(reloaded_facts_df) > 0
        and reloaded_facts_df[
            "issuer_id"
        ].notna().sum()
        == 0
    ):
        raise RuntimeError(
            "Persisted European facts contain "
            "no issuer_id values."
        )

    reloaded_filings_df = (
        reloaded_block_4_data[
            "europe_filing_metadata_df"
        ]
    )

    if (
        len(reloaded_filings_df) > 0
        and reloaded_filings_df[
            "issuer_id"
        ].notna().sum()
        == 0
    ):
        raise RuntimeError(
            "Persisted European filing metadata "
            "contains no issuer_id values."
        )

    block_4_validation_report_df = (
        pd.DataFrame(
            validation_rows
        )
    )

    display(
        block_4_validation_report_df
    )

    print(
        "Block 4 persistence validation passed. "
        "Downstream modules can load the issuer-centric "
        "European fundamentals layer without rerunning "
        "filing discovery or XBRL extraction."
    )

,table_name,expected_rows,persisted_rows,status
0,europe_coverage_report_df,13,13,PASSED
1,europe_economic_issuer_universe_df,77,77,PASSED
2,europe_entity_relationship_graph_df,0,0,PASSED
3,europe_filing_metadata_df,10281,10281,PASSED
4,europe_filing_system_registry_df,32,32,PASSED
5,europe_fundamentals_security_linked_df,13063,13063,PASSED
6,europe_fundamentals_standardised_df,11756,11756,PASSED
7,europe_issuer_security_link_quality_df,8,8,PASSED
8,europe_issuer_security_universe_df,81,81,PASSED
9,europe_lei_issuer_bridge_df,64,64,PASSED


Block 4 persistence validation passed. Downstream modules can load the issuer-centric European fundamentals layer without rerunning filing discovery or XBRL extraction.


## Architectural notes

European accounting facts remain issuer-level. A confirmed LEI-to-issuer bridge
controls filing linkage, while a separate audited table expands facts to listed
securities. Block 10 should load the current issuer-centric aliases.
